### This notebook extract newly generated molecules after the DORAnet run: 

In [50]:
import os
import io
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, deque
from io import BytesIO
from IPython.display import display
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, Draw, QED, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.SimDivFilters.rdSimDivPickers import MaxMinPicker

from PIL import Image, ImageDraw, ImageFont

RDLogger.DisableLog('rdApp.*')

In [51]:
from DORA_XGB import DORA_XGB
by_desc_MW_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'by_descending_MW')
by_asc_MW_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'by_ascending_MW')
add_concat_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_concat')
add_subtract_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_subtract')

In [52]:
fileNamePrefix = "basidalinPrecursor_gen2"

In [53]:
def canonSmiles(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol else str(smi)

### Discover All Starter Directories

In [54]:
doranetOutputDir = "doranet_output"
doranet_generations = 2

helpersToExclude = {
    'O', 'O=O', '[H][H]', 'O=C=O', 'C=O', '[C-]#[O+]', 'Br', '[Br][Br]',
    'CO', 'C=C', 'O=S(O)O', 'N', 'O=S(=O)(O)O', 'O=NO', 'N#N',
    'O=[N+]([O-])O', 'NO', 'C#N', 'S', 'O=S=O', 'N#CO', '[H+]', 'OO',
    'Cl', 'I', 'O=C(O)O', 'O=P(O)(O)O', 'O=P(O)(O)OP(=O)(O)O', 'C',
    'CC', 'CC=O', 'CC(=O)O', 'CCC(=O)O'
}

print(f"Output directory: {doranetOutputDir}")
print(f"Number of helpers to exclude: {len(helpersToExclude)}")

Output directory: doranet_output
Number of helpers to exclude: 33


In [55]:
# Automatically find all starter directories (exclude files)
allStarterDirPaths = sorted([
    p for p in glob.glob(os.path.join(doranetOutputDir, "starter_*"))
    if os.path.isdir(p)
])

print(f"Found {len(allStarterDirPaths)} starter directories")

# Find the molecules CSV file inside each directory
discoveredCsvFiles = []
directoriesWithMissingCsv = []

for starterDirPath in allStarterDirPaths:
    dirName = os.path.basename(starterDirPath)

    # Expected CSV file name matches directory name
    expectedCsvPath = os.path.join(starterDirPath, f"{dirName}_molecules.csv")

    if os.path.exists(expectedCsvPath):
        discoveredCsvFiles.append({
            'dirName': dirName,
            'dirPath': starterDirPath,
            'csvPath': expectedCsvPath,
            'starterNum': int(dirName.replace('starter_', ''))
        })
    else:
        # Try to find any molecules CSV in the directory as fallback
        fallbackCsvPaths = glob.glob(os.path.join(starterDirPath, "*_molecules.csv"))
        if fallbackCsvPaths:
            discoveredCsvFiles.append({
                'dirName': dirName,
                'dirPath': starterDirPath,
                'csvPath': fallbackCsvPaths[0],
                'starterNum': int(dirName.replace('starter_', ''))
            })
        else:
            directoriesWithMissingCsv.append(dirName)

# Sort by starter number
discoveredCsvFiles.sort(key=lambda x: x['starterNum'])

print(f"Found {len(discoveredCsvFiles)} CSV files containing DORAnet generated molecules")
if directoriesWithMissingCsv:
    print(f"Missing CSV in {len(directoriesWithMissingCsv)} directories: "
          f"{directoriesWithMissingCsv[:10]}...")

Found 2054 starter directories
Found 2054 CSV files containing DORAnet generated molecules


### Extract the `DORAnet` starter molecule

In [56]:
user_starters = set()
starterReadErrors = []

for fileInfo in discoveredCsvFiles:
    try:
        # Read only the SMILES and Is_Starter columns to save memory
        tempDF = pd.read_csv(
            fileInfo['csvPath'],
            usecols=['SMILES', 'Is_Starter']
        )
        starterSmiles = (
            tempDF[tempDF['Is_Starter'] == True]['SMILES']
            .dropna()
            .tolist()
        )
        user_starters.update(starterSmiles)

    except Exception as e:
        starterReadErrors.append({
            'dirName': fileInfo['dirName'],
            'csvPath': fileInfo['csvPath'],
            'error': str(e)
        })

if len(user_starters) == 0:
    raise ValueError("No starter molecules found across any CSV files!")

print(f"Number of unique starter molecules found: {len(user_starters)}")

Number of unique starter molecules found: 2054


### Read `reaction_strings` from the `DORAnet` generated `json` file

In [58]:
allStarterDirPaths = sorted([
    p for p in glob.glob(os.path.join(doranetOutputDir, "starter_*"))
    if os.path.isdir(p)
])
print(f"Found {len(allStarterDirPaths)} starter directories")

discoveredJsonFiles        = []
directoriesWithMissingJson = []

for starterDirPath in allStarterDirPaths:
    dirName          = os.path.basename(starterDirPath)
    expectedJsonPath = os.path.join(starterDirPath, f"{dirName}_network_pretreated.json")

    if os.path.exists(expectedJsonPath):
        discoveredJsonFiles.append({
            "dirName"   : dirName,
            "dirPath"   : starterDirPath,
            "jsonPath"  : expectedJsonPath,
            "starterNum": int(dirName.replace("starter_", "")),
        })
    else:
        fallbackJsonPaths = glob.glob(os.path.join(starterDirPath, "*_network_pretreated.json"))
        if fallbackJsonPaths:
            discoveredJsonFiles.append({
                "dirName"   : dirName,
                "dirPath"   : starterDirPath,
                "jsonPath"  : fallbackJsonPaths[0],
                "starterNum": int(dirName.replace("starter_", "")),
            })
        else:
            directoriesWithMissingJson.append(dirName)

discoveredJsonFiles.sort(key=lambda x: x["starterNum"])

print(f"Found {len(discoveredJsonFiles)} JSON files")
if directoriesWithMissingJson:
    print(f"Missing JSON in {len(directoriesWithMissingJson)} directories: "
          f"{directoriesWithMissingJson[:10]}")

for fileInfo in discoveredJsonFiles[:3]:
    print(f"  {fileInfo['dirName']} -> {os.path.basename(fileInfo['jsonPath'])}")
if len(discoveredJsonFiles) > 3:
    print(f"  ... and {len(discoveredJsonFiles) - 3} more")


def splitMoleculeString(moleculeString):
    if pd.isna(moleculeString) if hasattr(pd, "isna") else moleculeString is None:
        return []
    return [mol for mol in str(moleculeString).split(".") if mol]


reactionRecords = []
jsonReadErrors  = []

for fileInfo in discoveredJsonFiles:
    try:
        with open(fileInfo["jsonPath"], "r", encoding="utf-8") as f:
            reactionList = json.load(f)

        for rxn in reactionList:
            parts = str(rxn).split(">")
            if len(parts) != 4:
                continue

            reactants, ruleName, metaBlock, products = parts
            metaParts = (metaBlock.split("$") + [None, None, None, None])[:4]
            thermo, reactantStoich, productStoich, reactionType = metaParts

            reactantMols = splitMoleculeString(reactants.strip())
            productMols  = splitMoleculeString(products.strip())

            reactionRecords.append({
                "reactants"            : reactants.strip(),
                "products"             : products.strip(),
                "reactionString"       : f"{reactants.strip()} >> {products.strip()}",
                "ruleName"             : ruleName.strip(),
                "thermo"               : thermo,
                "reactantStoich"       : reactantStoich,
                "productStoich"        : productStoich,
                "reactionType"         : reactionType,
                "numReactantMolecules" : len(reactantMols),
                "numProductMolecules"  : len(productMols),
                "sourceStarterNum"     : fileInfo["starterNum"],
                "sourceDirectory"      : fileInfo["dirName"],
            })

    except Exception as e:
        jsonReadErrors.append({
            "dirName" : fileInfo["dirName"],
            "jsonPath": fileInfo["jsonPath"],
            "error"   : str(e),
        })
        print(f"  ERROR reading {fileInfo['dirName']}: {e}")

if not reactionRecords:
    raise RuntimeError("No reaction records loaded — check discoveredJsonFiles and JSON format.")

reactionDF = pd.DataFrame(reactionRecords).reset_index(drop=True)

allReactantMols = reactionDF["reactants"].apply(splitMoleculeString).explode()
allProductMols  = reactionDF["products"].apply(splitMoleculeString).explode()

print(f"JSON files read successfully         : {len(discoveredJsonFiles) - len(jsonReadErrors)}")
print(f"JSON files failed                    : {len(jsonReadErrors)}")
print(f"Total reactions                      : {len(reactionDF):,}")
print(f"Unique reactant strings              : {reactionDF['reactants'].nunique():,}")
print(f"Unique product strings               : {reactionDF['products'].nunique():,}")
print(f"Unique individual reactant molecules : {allReactantMols.nunique():,}")
print(f"Unique individual product molecules  : {allProductMols.nunique():,}")
print(f"Unique reaction rules                : {reactionDF['ruleName'].nunique():,}")
print(f"Unique source directories            : {reactionDF['sourceDirectory'].nunique():,}")
print(f"Reaction type counts:")
print(reactionDF["reactionType"].value_counts(dropna=False).to_string())

if jsonReadErrors:
    print(f"\nFailed files:")
    for err in jsonReadErrors:
        print(f"  {err['dirName']}: {err['error']}")

reactionDF.to_csv("reactionDF.csv", index=False)
print("\nSaved to: reactionDF.csv")

reactionDF

Found 2054 starter directories
Found 2054 JSON files
  starter_00000 -> starter_00000_network_pretreated.json
  starter_00001 -> starter_00001_network_pretreated.json
  starter_00002 -> starter_00002_network_pretreated.json
  ... and 2051 more
JSON files read successfully         : 2054
JSON files failed                    : 0
Total reactions                      : 2,022,023
Unique reactant strings              : 261,772
Unique product strings               : 1,134,192
Unique individual reactant molecules : 30,356
Unique individual product molecules  : 650,848
Unique reaction rules                : 476
Unique source directories            : 1,489
Reaction type counts:
reactionType
Enzymatic    2022023

Saved to: reactionDF.csv


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,sourceStarterNum,sourceDirectory
0,CCC=C(O)[C@H](O)[C@H](C)C(=O)O,C[C@@H]1C(O)OCCC=C(O)[C@@H]1O,CCC=C(O)[C@H](O)[C@H](C)C(=O)O >> C[C@@H]1C(O)...,rule0169_1,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,0,starter_00000
1,CCC(O)=C1OC(O)[C@@H](C)[C@H]1O.C[S+](CC[C@H](N...,CCC(O)=C1OC(OC)[C@@H](C)[C@H]1O.Nc1ncnc2c1ncn2...,CCC(O)=C1OC(O)[C@@H](C)[C@H]1O.C[S+](CC[C@H](N...,rule0011_51,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,0,starter_00000
2,CCC=C1OC(=O)[C@@H](C)[C@]1(C)O.C[S+](CC[C@H](N...,CCC(C)=C1OC(=O)[C@@H](C)[C@]1(C)O.Nc1ncnc2c1nc...,CCC=C1OC(=O)[C@@H](C)[C@]1(C)O.C[S+](CC[C@H](N...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,0,starter_00000
3,CCC(C(=O)O)=C1OC(=O)[C@@H](C)[C@H]1O.Cl,CCC(C(=O)Cl)=C1OC(=O)[C@@H](C)[C@H]1O.O,CCC(C(=O)O)=C1OC(=O)[C@@H](C)[C@H]1O.Cl >> CCC...,rule0183_6,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,0,starter_00000
4,CC(C)=C1OC(=O)[C@@H](C)[C@H]1O.O=C=O,CC(CC(=O)O)=C1OC(=O)[C@@H](C)[C@H]1O,CC(C)=C1OC(=O)[C@@H](C)[C@H]1O.O=C=O >> CC(CC(...,rule0023_17,No_Thermo,"(1, 1)","(1,)",Enzymatic,2,1,0,starter_00000
...,...,...,...,...,...,...,...,...,...,...,...,...
2022018,C[C@@H](C=O)C=C1C=C(O)C(=O)O1.O,C[C@](O)(C=O)C=C1C=C(O)C(O)O1,C[C@@H](C=O)C=C1C=C(O)C(=O)O1.O >> C[C@](O)(C=...,rule0070_6,No_Thermo,"(1, 1)","(1,)",Enzymatic,2,1,2053,starter_02053
2022019,C[C@H](C=C1C=C(O)C(=O)O1)CNC(N)[C@H](C)N.O,C[C@H](C=C1C=C(O)C(=O)O1)CNC(O)[C@H](C)N.N,C[C@H](C=C1C=C(O)C(=O)O1)CNC(N)[C@H](C)N.O >> ...,rule0056_18,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,2053,starter_02053
2022020,CC[C@H](C=C1C=C(O)C(=O)O1)CNC(=O)CN,CC(N)C(=O)NC[C@H](C)C=C1C=C(O)C(=O)O1,CC[C@H](C=C1C=C(O)C(=O)O1)CNC(=O)CN >> CC(N)C(...,rule0028_51,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,2053,starter_02053
2022021,CNC(=O)[C@@H](N)C[C@H](C)C=C1C=C(O)C(=O)O1.NC(...,CNC(=O)[C@@H](N)C[C@H](C)C1OC12C=C(O)C(=O)O2.N...,CNC(=O)[C@@H](N)C[C@H](C)C=C1C=C(O)C(=O)O1.NC(...,rule0077_4,No_Thermo,"(1, 1, 1)","(1, 1, 1)",Enzymatic,3,3,2053,starter_02053


### Identify `generated` compounds with `substructure` match with `starter` compound

In [65]:
COFACTOR_SMILES = [
    "NC(=O)c1ccc[n+](C2OC(COP(=O)([O-])OP(=O)([O-])OCC3OC(n4cnc5c(N)ncnc54)C(O)C3O)C(O)C2O)c1",
    "NC(=O)C1=CN(C2OC(COP(=O)([O-])OP(=O)([O-])OCC3OC(n4cnc5c(N)ncnc54)C(O)C3O)C(O)C2O)C=CC1",
    "NC(=O)c1ccc[n+](C2OC(COP(=O)([O-])OP(=O)([O-])OCC3OC(n4cnc5c(N)ncnc54)C(OP(=O)([O-])[O-])C3O)C(O)C2O)c1",
    "NC(=O)C1=CN(C2OC(COP(=O)([O-])OP(=O)([O-])OCC3OC(n4cnc5c(N)ncnc54)C(OP(=O)([O-])[O-])C3O)C(O)C2O)C=CC1",
    "Nc1ncnc2c1ncn2C1OC(COP(=O)(O)OP(=O)(O)OP(=O)(O)O)C(O)C1O",
    "Nc1ncnc2c1ncn2C1OC(COP(=O)(O)OP(=O)(O)O)C(O)C1O",
    "Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1O",
    "CC(C)(COP(=O)(O)OP(=O)(O)OCC1OC(n2cnc3c(N)ncnc32)C(O)C1OP(=O)(O)O)C(O)C(=O)NCCC(=O)NCCS",
    "Cc1cc2nc3c(=O)[nH]c(=O)nc3n(CC(O)C(O)C(O)COP(=O)(O)OP(=O)(O)OCC3OC(n4cnc5c(N)ncnc54)C(O)C3O)c2cc1C",
    "Cc1cc2nc3c(=O)[nH]c(=O)nc3n(CC(O)C(O)C(O)COP(=O)(O)O)c2cc1C",
    "C[S+](CCC(N)C(=O)[O-])CC1OC(n2cnc3c(N)ncnc32)C(O)C1O",
    "Nc1ncnc2c1ncn2C1OC(CS)C(O)C1O",
]

PHYSICOCHEMICAL_CUTOFFS = {
    "mw_max"  : 700.0,
    "tpsa_max": 200.0,
    "hbd_max" : 7,
    "qed_min" : 0.10,
}


def buildCofactorExclusionSet(rawSmilesList):
    exclusionSet = set()
    for smi in rawSmilesList:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            exclusionSet.add(Chem.MolToSmiles(mol))
        else:
            print(f"  Warning: could not parse cofactor SMILES: {smi}")
    print(f"Cofactor exclusion set built: {len(exclusionSet)} entries resolved.")
    return exclusionSet


def isCofactorOrEndogenous(mol, canonSmi, cofactorExclusionSet, cutoffs):
    if mol is None:
        return True
    if canonSmi in cofactorExclusionSet:
        return True
    mw   = Descriptors.MolWt(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    hbd  = rdMolDescriptors.CalcNumHBD(mol)
    if mw > cutoffs["mw_max"] or tpsa > cutoffs["tpsa_max"] or hbd > cutoffs["hbd_max"]:
        return True
    qed = QED.qed(mol)
    if qed < cutoffs["qed_min"]:
        return True
    return False


def hasStarterScaffold(mol, starterMols):
    if mol is None:
        return False, None
    for starterCanon, starterMol in starterMols:
        if mol.HasSubstructMatch(starterMol):
            return True, starterCanon
    return False, None


def buildStarterDerivedSet(reactionDF, starterMols, canonStarterSet,
                           canonHelperSet, cofactorExclusionSet, cutoffs):
    molToReactionIdxs = defaultdict(list)
    for idx, row in reactionDF.iterrows():
        for smi in splitMoleculeString(row["reactants"]):
            molToReactionIdxs[canonSmiles(smi)].append(idx)

    starterDerivedSet = {canonSmi: canonSmi for canonSmi, _ in starterMols}
    queue   = deque(canonStarterSet)
    visited = set(canonStarterSet)

    while queue:
        currentCanon = queue.popleft()
        for rxnIdx in molToReactionIdxs.get(currentCanon, []):
            row = reactionDF.loc[rxnIdx]
            for productSmi in splitMoleculeString(row["products"]):
                productCanon = canonSmiles(productSmi)

                if productCanon in visited:
                    continue
                visited.add(productCanon)

                if productCanon in canonHelperSet:
                    continue

                productMol = Chem.MolFromSmiles(str(productSmi))

                if isCofactorOrEndogenous(productMol, productCanon,
                                          cofactorExclusionSet, cutoffs):
                    continue

                isDerivative, matchedStarter = hasStarterScaffold(productMol, starterMols)
                if isDerivative:
                    starterDerivedSet[productCanon] = matchedStarter
                    queue.append(productCanon)

    return starterDerivedSet




def processSingleDirectory(args):
    """
    Top-level worker function for ProcessPoolExecutor.
    Must be top-level (not a closure) to be picklable by multiprocessing.
    All RDKit objects are reconstructed inside the worker from SMILES strings.
    """
    (dirName, starterSmilesList, rxnRecords,
     canonHelperSetArg, cofactorExclusionSetArg, cutoffsArg) = args

    from collections import defaultdict, deque
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors, QED

    def localSplit(s):
        if s is None:
            return []
        return [m for m in str(s).split(".") if m]

    def localCanon(smi):
        mol = Chem.MolFromSmiles(str(smi))
        return Chem.MolToSmiles(mol) if mol else str(smi)

    def localIsExcluded(mol, canonSmi):
        if mol is None:
            return True
        if canonSmi in cofactorExclusionSetArg:
            return True
        if Descriptors.MolWt(mol)        > cutoffsArg["mw_max"]:
            return True
        if rdMolDescriptors.CalcTPSA(mol) > cutoffsArg["tpsa_max"]:
            return True
        if rdMolDescriptors.CalcNumHBD(mol) > cutoffsArg["hbd_max"]:
            return True
        return QED.qed(mol) < cutoffsArg["qed_min"]

    starterMols = [
        (localCanon(smi), Chem.MolFromSmiles(str(smi)))
        for smi in starterSmilesList
        if Chem.MolFromSmiles(str(smi)) is not None
    ]
    if not starterMols:
        return []

    canonStarterSet = {s for s, _ in starterMols}

    molToRxnIdxs = defaultdict(list)
    for i, rec in enumerate(rxnRecords):
        for smi in localSplit(rec["reactants"]):
            molToRxnIdxs[localCanon(smi)].append(i)

    starterDerivedSet = {s: s for s in canonStarterSet}
    queue   = deque(canonStarterSet)
    visited = set(canonStarterSet)

    while queue:
        currentCanon = queue.popleft()
        for rxnIdx in molToRxnIdxs.get(currentCanon, []):
            for productSmi in localSplit(rxnRecords[rxnIdx]["products"]):
                productCanon = localCanon(productSmi)
                if productCanon in visited:
                    continue
                visited.add(productCanon)
                if productCanon in canonHelperSetArg:
                    continue
                productMol = Chem.MolFromSmiles(str(productSmi))
                if localIsExcluded(productMol, productCanon):
                    continue
                for starterCanon, starterMol in starterMols:
                    if productMol.HasSubstructMatch(starterMol):
                        starterDerivedSet[productCanon] = starterCanon
                        queue.append(productCanon)
                        break

    allProductSmis = set()
    for rec in rxnRecords:
        for smi in localSplit(rec["products"]):
            allProductSmis.add(smi)

    records = []
    for smi in allProductSmis:
        mol      = Chem.MolFromSmiles(str(smi))
        canonSmi = Chem.MolToSmiles(mol) if mol else None
        if canonSmi is None:
            continue
        if canonSmi in canonHelperSetArg:
            continue
        if canonSmi in canonStarterSet:
            continue
        if localIsExcluded(mol, canonSmi):
            continue
        if canonSmi not in starterDerivedSet:
            continue
        matchedRaw = starterDerivedSet.get(canonSmi)
        records.append({
            "sourceDirectory"          : dirName,
            "starter_Canonical_SMILES" : localCanon(matchedRaw) if matchedRaw else None,
            "Canonical_SMILES"         : canonSmi,
            "molecularFormula"         : rdMolDescriptors.CalcMolFormula(mol),
            "molecularWeight"          : round(Descriptors.MolWt(mol), 4),
            "numHeavyAtoms"            : mol.GetNumHeavyAtoms(),
        })
    return records


COFACTOR_EXCLUSION_SET = buildCofactorExclusionSet(COFACTOR_SMILES)
canonHelperSet         = {canonSmiles(s) for s in helpersToExclude}

dirToStarterSmis  = {}
starterReadErrors = []

for fileInfo in discoveredCsvFiles:
    try:
        tempDF = pd.read_csv(fileInfo["csvPath"], usecols=["SMILES", "Is_Starter"])
        starterSmilesList = (
            tempDF[tempDF["Is_Starter"] == True]["SMILES"]
            .dropna()
            .tolist()
        )
        if starterSmilesList:
            dirToStarterSmis[fileInfo["dirName"]] = starterSmilesList
    except Exception as e:
        starterReadErrors.append({"dirName": fileInfo["dirName"], "error": str(e)})

print(f"Directories with identified starters : {len(dirToStarterSmis):,}")
print(f"Directories with read errors         : {len(starterReadErrors):,}")

print("Pre-grouping reactions by source directory (runs once)...")
dirToRxnRecords = {
    dirName: grp.to_dict("records")
    for dirName, grp in reactionDF.groupby("sourceDirectory")
}
print(f"Pre-grouped {len(dirToRxnRecords):,} directories.")

argsList = [
    (
        dirName,
        starterSmilesList,
        dirToRxnRecords.get(dirName, []),
        canonHelperSet,
        COFACTOR_EXCLUSION_SET,
        PHYSICOCHEMICAL_CUTOFFS,
    )
    for dirName, starterSmilesList in dirToStarterSmis.items()
    if dirToRxnRecords.get(dirName)
]

nWorkers = min(multiprocessing.cpu_count(), len(argsList))
print(f"Processing {len(argsList)} directories with {nWorkers} parallel workers...")

allProductRecords = []
nCompleted        = 0
logInterval       = max(1, len(argsList) // 10)

with ProcessPoolExecutor(max_workers=nWorkers) as executor:
    futures = {
        executor.submit(processSingleDirectory, args): args[0]
        for args in argsList
    }
    for future in as_completed(futures):
        dirName = futures[future]
        try:
            records = future.result()
            allProductRecords.extend(records)
        except Exception as e:
            print(f"  ERROR in {dirName}: {e}")
        nCompleted += 1
        if nCompleted % logInterval == 0 or nCompleted == len(argsList):
            print(f"  Completed {nCompleted:,} / {len(argsList):,} directories "
                  f"({nCompleted / len(argsList) * 100:.0f}%)  "
                  f"— {len(allProductRecords):,} compounds collected so far...")

generatedCompoundsDF_raw = (
    pd.DataFrame(allProductRecords)
    .sort_values(
        ["starter_Canonical_SMILES", "molecularWeight"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

nBeforeDedup = len(generatedCompoundsDF_raw)

generatedCompoundsDF = (
    generatedCompoundsDF_raw
    .drop_duplicates(subset="Canonical_SMILES", keep="first")
    .reset_index(drop=True)
)

nAfterDedup = len(generatedCompoundsDF)

generatedCompoundsCsvPath = f"{fileNamePrefix}_generated_compounds.csv"
generatedCompoundsDF.to_csv(generatedCompoundsCsvPath, index=False)

print(f"Compounds before deduplication        : {nBeforeDedup:,}")
print(f"Compounds after deduplication         : {nAfterDedup:,}")
print(f"Duplicates removed                    : {nBeforeDedup - nAfterDedup:,}")
print(f"\nSaved to: {generatedCompoundsCsvPath}")

generatedCompoundsDF

Cofactor exclusion set built: 12 entries resolved.
Directories with identified starters : 2,054
Directories with read errors         : 0
Pre-grouping reactions by source directory (runs once)...
Pre-grouped 1,489 directories.
Processing 1489 directories with 112 parallel workers...
  Completed 148 / 1,489 directories (10%)  — 28,061 compounds collected so far...
  Completed 296 / 1,489 directories (20%)  — 63,641 compounds collected so far...
  Completed 444 / 1,489 directories (30%)  — 93,432 compounds collected so far...
  Completed 592 / 1,489 directories (40%)  — 99,711 compounds collected so far...
  Completed 740 / 1,489 directories (50%)  — 104,269 compounds collected so far...
  Completed 888 / 1,489 directories (60%)  — 116,631 compounds collected so far...
  Completed 1,036 / 1,489 directories (70%)  — 127,165 compounds collected so far...
  Completed 1,184 / 1,489 directories (80%)  — 132,206 compounds collected so far...
  Completed 1,332 / 1,489 directories (89%)  — 136,74

,sourceDirectory,starter_Canonical_SMILES,Canonical_SMILES,molecularFormula,molecularWeight,numHeavyAtoms
0,starter_00519,C=CCC1=CC(=CC(C)C)OC1=O,C=CC(=O)C1=CC(=CC(C)C)OC1=O,C11H12O3,192.214,14
1,starter_00519,C=CCC1=CC(=CC(C)C)OC1=O,C=CCC1=CC(=CC(C)C=O)OC1=O,C11H12O3,192.214,14
2,starter_00519,C=CCC1=CC(=CC(C)C)OC1=O,C=CC(C)C1=CC(=CC(C)C)OC1=O,C12H16O2,192.258,14
3,starter_00519,C=CCC1=CC(=CC(C)C)OC1=O,C=C(C)CC1=CC(=CC(C)C)OC1=O,C12H16O2,192.258,14
4,starter_00519,C=CCC1=CC(=CC(C)C)OC1=O,C=CCC1=CC(=CC(C)(C)C)OC1=O,C12H16O2,192.258,14
...,...,...,...,...,...,...
130509,starter_01804,O=C1OC(=Cc2ccccc2)[C@H](O)[C@H]1O,COc1cccc(C=C2OC(=O)[C@H](O)[C@H]2O)c1,C12H12O5,236.223,17
130510,starter_01804,O=C1OC(=Cc2ccccc2)[C@H](O)[C@H]1O,CO[C@H]1C(=Cc2ccc(O)cc2)OC(=O)[C@@H]1O,C12H12O5,236.223,17
130511,starter_01804,O=C1OC(=Cc2ccccc2)[C@H](O)[C@H]1O,CO[C@H]1C(=Cc2ccccc2O)OC(=O)[C@@H]1O,C12H12O5,236.223,17
130512,starter_01804,O=C1OC(=Cc2ccccc2)[C@H](O)[C@H]1O,COC(=C1OC(=O)[C@H](O)[C@H]1O)c1ccccc1,C12H12O5,236.223,17


### Select `structurally` diverse compounds from the generated compound list

In [ ]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

nDiverseSelect = 100000
nWorkers       = 8
batchSize      = 10000
preSampleCap   = 300000
fpRadius       = 2
fpNBits        = 2048


def computeFpBatch(args):
    """
    Top-level worker: compute Morgan fingerprints for one batch.
    Returns list of (original_index, ExplicitBitVect).
    Must be top-level for multiprocessing pickling.
    """
    from rdkit import Chem
    from rdkit.Chem import AllChem
    indexedSmiles, radius, nBits = args
    results = []
    for origIdx, smi in indexedSmiles:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)
            results.append((origIdx, fp))
    return results


smilesList = generatedCompoundsDF["Canonical_SMILES"].tolist()
nTotal     = len(smilesList)
print(f"Total compounds in generatedCompoundsDF : {nTotal:,}")

if nTotal > preSampleCap:
    print(f"Pool exceeds {preSampleCap:,}. Performing stratified pre-sample "
          f"before MaxMinPicker...")
    rng          = np.random.default_rng(seed=42)
    preSampleIdx = rng.choice(nTotal, size=preSampleCap, replace=False)
    preSampleIdx = sorted(preSampleIdx.tolist())
    candidateSmiles = [(i, smilesList[i]) for i in preSampleIdx]
    print(f"Pre-sample size : {len(candidateSmiles):,}")
else:
    candidateSmiles = list(enumerate(smilesList))
    preSampleIdx    = list(range(nTotal))

batches = [
    (candidateSmiles[i : i + batchSize], fpRadius, fpNBits)
    for i in range(0, len(candidateSmiles), batchSize)
]

print(f"Computing Morgan fingerprints across {nWorkers} workers "
      f"({len(batches)} batches of {batchSize:,})...")

localIdxToFp   = {}
localIdxToOrig = {}

with ProcessPoolExecutor(max_workers=nWorkers) as executor:
    for batchResult in executor.map(computeFpBatch, batches):
        for origIdx, fp in batchResult:
            localIdx                 = len(localIdxToFp)
            localIdxToFp[localIdx]   = fp
            localIdxToOrig[localIdx] = origIdx

nValid  = len(localIdxToFp)
nSelect = min(nDiverseSelect, nValid)
fpsList = [localIdxToFp[i] for i in range(nValid)]

print(f"Valid fingerprints computed : {nValid:,}")
print(f"Selecting diverse compounds : {nSelect:,}")
print("Running MaxMinPicker with LazyBitVectorPick (C++ Tanimoto, no Python callback)...")

picker          = MaxMinPicker()
pickedLocalIdxs = list(picker.LazyBitVectorPick(fpsList, nValid, nSelect, seed=42))
pickedLocalSet  = set(pickedLocalIdxs)
pickedDfIdxs    = sorted([localIdxToOrig[k] for k in pickedLocalIdxs])

print(f"Diversity selection complete. Building fingerprint matrix for UMAP...")

fpMatrix = np.zeros((nValid, fpNBits), dtype=np.uint8)
for i, fp in enumerate(fpsList):
    DataStructs.ConvertToNumpyArray(fp, fpMatrix[i])

nBefore              = len(generatedCompoundsDF)
generatedCompoundsDF = (
    generatedCompoundsDF
    .iloc[pickedDfIdxs]
    .copy()
    .reset_index(drop=True)
)
print(f"generatedCompoundsDF before diversity selection : {nBefore:,}")
print(f"generatedCompoundsDF after  diversity selection : {len(generatedCompoundsDF):,}")

print(f"Running UMAP with {nWorkers} parallel jobs...")
try:
    import umap
    reducer    = umap.UMAP(
        n_components = 2,
        random_state = 42,
        metric       = "jaccard",
        n_neighbors  = 15,
        min_dist     = 0.1,
        n_jobs       = nWorkers,
        low_memory   = True,
    )
    embedding  = reducer.fit_transform(fpMatrix)
    methodName = "UMAP"
except ImportError:
    from sklearn.manifold import TSNE
    print("umap-learn not found, falling back to t-SNE (slower for large n).")
    reducer    = TSNE(
        n_components = 2,
        random_state = 42,
        metric       = "jaccard",
        perplexity   = 40,
        n_iter       = 1000,
        n_jobs       = nWorkers,
    )
    embedding  = reducer.fit_transform(fpMatrix)
    methodName = "t-SNE"

print(f"Dimensionality reduction complete: {methodName}")

notPickedLocalIdxs = [i for i in range(nValid) if i not in pickedLocalSet]

maxVizPoints = 50000
if len(notPickedLocalIdxs) > maxVizPoints:
    rng              = np.random.default_rng(seed=0)
    notPickedLocalIdxs = rng.choice(
        notPickedLocalIdxs, size=maxVizPoints, replace=False
    ).tolist()

unselX    = embedding[notPickedLocalIdxs, 0]
unselY    = embedding[notPickedLocalIdxs, 1]
divLocalX = embedding[list(pickedLocalIdxs), 0]
divLocalY = embedding[list(pickedLocalIdxs), 1]

fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(
    unselX, unselY,
    c          = "#2ecc71",
    marker     = "o",
    s          = 8,
    alpha      = 0.3,
    linewidths = 0,
    label      = f"Not selected (n={nValid - nSelect:,})",
    zorder     = 2,
    rasterized = True,
)
ax.scatter(
    divLocalX, divLocalY,
    c          = "#e74c3c",
    marker     = "*",
    s          = 30,
    alpha      = 0.8,
    linewidths = 0.2,
    edgecolors = "#c0392b",
    label      = f"Diverse selected (n={nSelect:,})",
    zorder     = 3,
    rasterized = True,
)

ax.set_xlabel(f"{methodName} Dimension 1", fontsize=10, fontweight="bold")
ax.set_ylabel(f"{methodName} Dimension 2", fontsize=10, fontweight="bold")
ax.legend(fontsize=10, markerscale=1.4, frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(labelsize=11)
plt.tight_layout()

plotPngPath = f"{fileNamePrefix}_chemical_space.png"
plotPdfPath = f"{fileNamePrefix}_chemical_space.pdf"
fig.savefig(plotPngPath, dpi=600, bbox_inches="tight")
fig.savefig(plotPdfPath, dpi=600, bbox_inches="tight")
print(f"Plot saved: {plotPngPath}")
print(f"Plot saved: {plotPdfPath}")
plt.show()

generatedCompoundsCsvPath = f"{fileNamePrefix}_generated_compounds.csv"
generatedCompoundsDF.to_csv(generatedCompoundsCsvPath, index=False)
print(f"\nSaved to: {generatedCompoundsCsvPath}")

Total compounds in generatedCompoundsDF : 130,514
Computing Morgan fingerprints across 8 workers (14 batches of 10,000)...


### Visualize `product` compounds

In [ ]:
nTargetsTotal   = 18
nTargetsPerRow  = 3
molCellW        = 420
molCellH        = 320
labelH          = 40
arrowZoneW      = 54
targetGap       = 10
dpi             = 600

white           = (255, 255, 255)
black           = (0,   0,   0  )
arrowColor      = (80,  80,  80 )

fontBoldPath    = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"


def loadFont(path, size):
    try:
        return ImageFont.truetype(path, size)
    except Exception:
        return ImageFont.load_default()


def renderMolToPil(mol, w, h):
    try:
        drawer = rdMolDraw2D.MolDraw2DCairo(w, h)
        drawer.drawOptions().addStereoAnnotation = True
        drawer.drawOptions().padding = 0.12
        rdMolDraw2D.PrepareMolForDrawing(mol)
        drawer.DrawMolecule(mol)
        drawer.FinishDrawing()
        return Image.open(io.BytesIO(drawer.GetDrawingText())).convert("RGB")
    except Exception:
        from rdkit.Chem import Draw
        return Draw.MolToImage(mol, size=(w, h))


def selectDiverseIdxs(smilesList, nSelect, seed=42):
    fps, validIdxs = [], []
    for i, smi in enumerate(smilesList):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fps.append(AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048))
            validIdxs.append(i)

    nSelect = min(nSelect, len(fps))

    def distFunc(i, j, fps=fps):
        return 1.0 - DataStructs.TanimotoSimilarity(fps[i], fps[j])

    picker      = MaxMinPicker()
    pickedLocal = list(picker.LazyPick(distFunc, len(fps), nSelect, seed=seed))
    return sorted([validIdxs[k] for k in pickedLocal])


def drawMolCell(mol, cellW, cellH, molH, label, fontBold):
    cell = Image.new("RGB", (cellW, cellH), white)
    ctx  = ImageDraw.Draw(cell)

    if mol is not None:
        molImg = renderMolToPil(mol, cellW - 16, molH - 8)
        cell.paste(molImg, (8, 4))

    bb  = ctx.textbbox((0, 0), label, font=fontBold)
    lW  = bb[2] - bb[0]
    ctx.text(((cellW - lW) // 2, molH + 10), label, fill=black, font=fontBold)
    return cell


def pasteArrow(canvas, x, y, length=34, color=arrowColor):
    d = ImageDraw.Draw(canvas)
    d.line([(x, y), (x + length, y)], fill=color, width=3)
    headL, headW = 10, 5
    d.polygon([
        (x + length,          y),
        (x + length - headL,  y - headW),
        (x + length - headL,  y + headW),
    ], fill=color)


def buildRowCanvas(starterMol, targetMols, fontBold):
    cellH = molCellH + labelH
    rowW  = (molCellW + arrowZoneW +
             nTargetsPerRow * molCellW +
             (nTargetsPerRow - 1) * targetGap)

    canvas      = Image.new("RGB", (rowW, cellH), white)
    starterCell = drawMolCell(starterMol, molCellW, cellH, molCellH, "Starter", fontBold)
    canvas.paste(starterCell, (0, 0))

    pasteArrow(canvas, x=molCellW + 6, y=cellH // 2, length=arrowZoneW - 16)

    for tIdx, targetMol in enumerate(targetMols):
        cellX      = molCellW + arrowZoneW + tIdx * (molCellW + targetGap)
        targetCell = drawMolCell(targetMol, molCellW, cellH, molCellH,
                                 "Generated Compound", fontBold)
        canvas.paste(targetCell, (cellX, 0))

    return canvas


print("Selecting structurally diverse compounds via MaxMinPicker...")
diverseIdxs = selectDiverseIdxs(
    generatedCompoundsDF["Canonical_SMILES"].tolist(),
    nTargetsTotal,
)
selectedDF = (
    generatedCompoundsDF
    .iloc[diverseIdxs]
    .copy()
    .reset_index(drop=True)
)
print(f"Selected {len(selectedDF)} compounds from {len(generatedCompoundsDF):,} total.")

rows = []
for starterSmi, grp in selectedDF.groupby("starter_Canonical_SMILES", sort=False):
    targets = grp["Canonical_SMILES"].tolist()
    for i in range(0, len(targets), nTargetsPerRow):
        batch = targets[i : i + nTargetsPerRow]
        while len(batch) < nTargetsPerRow:
            batch.append(None)
        rows.append({"starterSmi": starterSmi, "targets": batch})

fontBold    = loadFont(fontBoldPath, 16)
rowCanvases = []

for rowIdx, rowData in enumerate(rows):
    starterMol = Chem.MolFromSmiles(rowData["starterSmi"])
    targetMols = [
        Chem.MolFromSmiles(smi) if smi else None
        for smi in rowData["targets"]
    ]

    rowCanvas = buildRowCanvas(starterMol, targetMols, fontBold)
    rowCanvases.append(rowCanvas)

    pngPath = f"{fileNamePrefix}_row{rowIdx + 1:02d}.png"
    pdfPath = f"{fileNamePrefix}_row{rowIdx + 1:02d}.pdf"
    rowCanvas.save(pngPath, format="PNG", dpi=(dpi, dpi))
    rowCanvas.save(pdfPath, format="PDF", resolution=dpi)
    print(f"Row {rowIdx + 1:02d} saved: {pngPath}  |  {pdfPath}")

rowGap     = 14
rowCellH   = molCellH + labelH
fullW      = rowCanvases[0].width
fullH      = len(rowCanvases) * rowCellH + (len(rowCanvases) - 1) * rowGap
fullCanvas = Image.new("RGB", (fullW, fullH), white)

for i, rc in enumerate(rowCanvases):
    fullCanvas.paste(rc, (0, i * (rowCellH + rowGap)))

print(f"\n{len(rows)} row figures saved individually at {dpi} DPI.")
fullCanvas

### Print `pathway` level details

In [ ]:
def buildReactionIndex(reactionDF, canonHelperSet):
    molToRxnIdxs = defaultdict(list)
    rxnProducts  = {}

    for idx, row in reactionDF.iterrows():
        for smi in splitMoleculeString(row["reactants"]):
            c = canonSmiles(smi)
            if c not in canonHelperSet:
                molToRxnIdxs[c].append(idx)

        rxnProducts[idx] = frozenset(
            canonSmiles(s) for s in splitMoleculeString(row["products"])
            if canonSmiles(s) not in canonHelperSet
        )

    return molToRxnIdxs, rxnProducts


def countChainsToTargets(starterCanon, targetCanonSet, molToRxnIdxs,
                         rxnProducts, starterMols, maxDepth=3):
    scaffoldCache = {}

    def isScaffoldBearing(canonSmi):
        if canonSmi not in scaffoldCache:
            mol = Chem.MolFromSmiles(canonSmi)
            scaffoldCache[canonSmi] = (
                mol is not None and
                any(mol.HasSubstructMatch(starterMol) for _, starterMol in starterMols)
            )
        return scaffoldCache[canonSmi]

    def productsHaveScaffold(products):
        return any(isScaffoldBearing(m) for m in products)

    counts        = defaultdict(lambda: defaultdict(int))
    queue         = deque()
    statesVisited = 0

    for rxnIdx in molToRxnIdxs.get(starterCanon, []):
        products = rxnProducts[rxnIdx]
        chain    = (rxnIdx,)

        for target in products & targetCanonSet:
            counts[target][1] += 1

        if maxDepth > 1:
            queue.append((chain, products, frozenset({rxnIdx})))

    while queue:
        chain, availMols, visitedRxns = queue.popleft()
        statesVisited += 1
        depth = len(chain)

        if depth >= maxDepth:
            continue

        for mol in availMols:
            for rxnIdx in molToRxnIdxs.get(mol, []):
                if rxnIdx in visitedRxns:
                    continue

                products  = rxnProducts[rxnIdx]
                newChain  = chain + (rxnIdx,)
                newDepth  = len(newChain)

                for target in products & targetCanonSet:
                    counts[target][newDepth] += 1

                if newDepth < maxDepth:
                    newState = (newChain, products, visitedRxns | {rxnIdx})
                    if productsHaveScaffold(products):
                        queue.appendleft(newState)
                    else:
                        queue.append(newState)

    return counts, statesVisited


print("Building reaction index...")
molToRxnIdxs, rxnProducts = buildReactionIndex(reactionDF, canonHelperSet)
print(f"Indexed {len(molToRxnIdxs):,} unique non-helper reactant molecules.")
print(f"Indexed {len(rxnProducts):,} reactions.\n")

totalCompounds = len(generatedCompoundsDF)
print(f"Total compounds after deduplication (generatedCompoundsDF): {totalCompounds:,}\n")

summaryRecords     = []
perStarterSummary  = []

for starterSmi, group in generatedCompoundsDF.groupby("starter_Canonical_SMILES"):
    targetCanonSet  = set(group["Canonical_SMILES"].tolist())
    nTargets        = len(targetCanonSet)

    print(f"Starter : {starterSmi[:70]}")
    print(f"  Compounds from this starter  : {nTargets:,}")

    counts, statesVisited = countChainsToTargets(
        starterSmi,
        targetCanonSet,
        molToRxnIdxs,
        rxnProducts,
        starterMols = starterMols,
        maxDepth    = doranet_generations,
    )

    nWithPath    = sum(1 for t in targetCanonSet if counts.get(t))
    nWithoutPath = nTargets - nWithPath
    coverage     = nWithPath / nTargets * 100 if nTargets > 0 else 0.0

    print(f"  BFS states explored          : {statesVisited:,}")
    print(f"  With atlest one pathway              : {nWithPath:,}  ({coverage:.1f}%)")
    print(f"  With zero pathways           : {nWithoutPath:,}")
    print()

    perStarterSummary.append({
        "starter_Canonical_SMILES" : starterSmi,
        "total_compounds"          : nTargets,
        "compounds_with_pathway"   : nWithPath,
        "compounds_without_pathway": nWithoutPath,
        "pathway_coverage_pct"     : round(coverage, 2),
    })

    for _, row in group.iterrows():
        targetSmi    = row["Canonical_SMILES"]
        targetCounts = counts.get(targetSmi, {})
        n1 = targetCounts.get(1, 0)
        n2 = targetCounts.get(2, 0)
        n3 = targetCounts.get(3, 0)

        summaryRecords.append({
            "starter_Canonical_SMILES" : starterSmi,
            "Canonical_SMILES"         : targetSmi,
            "1step_pathways"           : n1,
            "2step_pathways"           : n2,
            "3step_pathways"           : n3,
            "total_pathways"           : n1 + n2 + n3,
        })

pathSummaryDF = (
    pd.DataFrame(summaryRecords)
    .sort_values(
        ["starter_Canonical_SMILES", "total_pathways"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

perStarterSummaryDF = pd.DataFrame(perStarterSummary)

nZero    = (pathSummaryDF["total_pathways"] == 0).sum()
nNonZero = (pathSummaryDF["total_pathways"] >  0).sum()


if nZero > 0:
    print(f"\n  Note: {nZero} zero-pathway compounds persist after exhaustive BFS.")
    print(f"  This indicates a canonicalization mismatch between")
    print(f"  buildStarterDerivedSet and countChainsToTargets.")

pathSummaryDF

### Visualize `starter → product` reaction pathways from the network

In [ ]:
molSize      = (260, 200)
sepWidth     = 80
stepBannerH  = 32
stepGap      = 10
depthWords   = {1: "One-Step", 2: "Two-Step", 3: "Three-Step"}


def loadFont(size):
    for path in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            continue
    return ImageFont.load_default()

fontSep   = loadFont(52)
fontLabel = loadFont(18)
fontStep  = loadFont(17)


def rdkitImageToPil(imageObj):
    if isinstance(imageObj, Image.Image):
        return imageObj.convert("RGB")
    if isinstance(imageObj, (bytes, bytearray)):
        return Image.open(BytesIO(imageObj)).convert("RGB")
    return Image.open(BytesIO(imageObj.data)).convert("RGB")


def centeredText(draw, text, imgWidth, y, font, color="black"):
    try:
        bb    = draw.textbbox((0, 0), text, font=font)
        textW = bb[2] - bb[0]
    except Exception:
        textW = len(text) * 8
    draw.text(((imgWidth - textW) // 2, y), text, fill=color, font=font)


def getMolLabel(canon, canonStarterSet, genuineProductSet, isLastStep, isProductSide):
    if not isProductSide:
        return "Starter" if canon in canonStarterSet else "Reactant"
    if isProductSide and canon in genuineProductSet and isLastStep:
        return "Target Compound"
    return "Product"


def makeLabeledMolImage(smi, label):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    molImg = rdkitImageToPil(Draw.MolToImage(mol, size=molSize))
    banner = Image.new("RGB", (molImg.width, 32), "#f0f0f0")
    centeredText(ImageDraw.Draw(banner), label, molImg.width, 6, fontLabel, "#222222")
    out = Image.new("RGB", (molImg.width, molImg.height + banner.height), "white")
    out.paste(molImg, (0, 0))
    out.paste(banner, (0, molImg.height))
    return out


def makeSeparatorImage(text, height, width=80):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        bb     = draw.textbbox((0, 0), text, font=fontSep)
        tw, th = bb[2] - bb[0], bb[3] - bb[1]
    except Exception:
        tw, th = 30, 50
    draw.text(((width - tw) // 2, (height - th) // 2), text, fill="#222222", font=fontSep)
    return img


def renderReactionStep(rxnIdx, reactionDF, canonStarterSet, genuineProductSet,
                       canonHelperSet, isLastStep, stepLabel):
    row = reactionDF.loc[rxnIdx]

    reactantSmis = splitMoleculeString(row["reactants"])
    productSmis  = splitMoleculeString(row["products"])

    reactantSmis = (
        [s for s in reactantSmis if canonSmiles(s) in canonStarterSet] +
        [s for s in reactantSmis if canonSmiles(s) not in canonStarterSet]
    )
    productSmis = (
        [s for s in productSmis if canonSmiles(s) in genuineProductSet] +
        [s for s in productSmis if canonSmiles(s) not in genuineProductSet]
    )

    reactantImgs = [
        img for img in [
            makeLabeledMolImage(
                s,
                getMolLabel(canonSmiles(s), canonStarterSet, genuineProductSet,
                            isLastStep, isProductSide=False),
            ) for s in reactantSmis
        ] if img is not None
    ]
    productImgs = [
        img for img in [
            makeLabeledMolImage(
                s,
                getMolLabel(canonSmiles(s), canonStarterSet, genuineProductSet,
                            isLastStep, isProductSide=True),
            ) for s in productSmis
        ] if img is not None
    ]

    if not reactantImgs or not productImgs:
        return None

    maxH  = max(img.height for img in reactantImgs + productImgs)
    gap   = 14
    parts = []
    for i, img in enumerate(reactantImgs):
        parts.append(img)
        if i < len(reactantImgs) - 1:
            parts.append(makeSeparatorImage("+", maxH, width=sepWidth))
    parts.append(makeSeparatorImage("→", maxH, width=100))
    for i, img in enumerate(productImgs):
        parts.append(img)
        if i < len(productImgs) - 1:
            parts.append(makeSeparatorImage("+", maxH, width=sepWidth))

    totalW = sum(p.width for p in parts) + gap * (len(parts) - 1)
    rxnRow = Image.new("RGB", (totalW, maxH), "white")
    x      = 0
    for part in parts:
        rxnRow.paste(part, (x, (maxH - part.height) // 2))
        x += part.width + gap

    stepBanner = Image.new("RGB", (totalW, stepBannerH), "#dce8f5")
    centeredText(ImageDraw.Draw(stepBanner), stepLabel, totalW,
                 (stepBannerH - 20) // 2, fontStep, "#003366")

    panel = Image.new("RGB", (totalW, stepBannerH + rxnRow.height), "white")
    panel.paste(stepBanner, (0, 0))
    panel.paste(rxnRow,     (0, stepBannerH))
    return panel


def stackStepPanels(panels):
    if not panels:
        return None
    w      = max(p.width for p in panels)
    totalH = sum(p.height for p in panels) + stepGap * (len(panels) - 1)
    canvas = Image.new("RGB", (w, totalH), "white")
    y = 0
    for p in panels:
        canvas.paste(p, (0, y))
        y += p.height + stepGap
    return canvas


def findPathwayChains(reactionDF, canonStarterSet, genuineProductSet,
                      canonHelperSet, maxDepth=3, maxPerDepth=3):
    """
    Exhaustive BFS with no frontier cap. Safe because DORAnet ran a bounded
    3-step expansion, so the reaction network is finite and the BFS is
    guaranteed to terminate.
    """
    molToRxnIdxs = defaultdict(list)
    for idx, row in reactionDF.iterrows():
        for smi in splitMoleculeString(row["reactants"]):
            c = canonSmiles(smi)
            if c not in canonHelperSet:
                molToRxnIdxs[c].append(idx)

    def getNonHelperProducts(rxnIdx):
        row = reactionDF.loc[rxnIdx]
        return frozenset(
            canonSmiles(s) for s in splitMoleculeString(row["products"])
            if canonSmiles(s) not in canonHelperSet
        )

    pathways = {d: [] for d in range(1, maxDepth + 1)}
    queue    = deque()

    for starterCanon in canonStarterSet:
        for rxnIdx in molToRxnIdxs.get(starterCanon, []):
            products = getNonHelperProducts(rxnIdx)
            chain    = (rxnIdx,)
            if products & genuineProductSet and len(pathways[1]) < maxPerDepth:
                pathways[1].append(chain)
            if maxDepth > 1:
                queue.append((chain, products, frozenset({rxnIdx})))

    while queue:
        chain, availMols, visitedRxns = queue.popleft()
        depth = len(chain)

        if depth >= maxDepth:
            continue

        if all(len(pathways[d]) >= maxPerDepth for d in range(depth + 1, maxDepth + 1)):
            continue

        for mol in availMols:
            for rxnIdx in molToRxnIdxs.get(mol, []):
                if rxnIdx in visitedRxns:
                    continue

                products  = getNonHelperProducts(rxnIdx)
                newChain  = chain + (rxnIdx,)
                newDepth  = len(newChain)

                if products & genuineProductSet and len(pathways[newDepth]) < maxPerDepth:
                    pathways[newDepth].append(newChain)

                if newDepth < maxDepth:
                    queue.append((newChain, products, visitedRxns | {rxnIdx}))

    return pathways


genuineProductSet = set(generatedCompoundsDF["Canonical_SMILES"].dropna().unique())

print("Searching for 1-step, 2-step, and 3-step pathway chains...")
pathwaysByDepth = findPathwayChains(
    reactionDF,
    canonStarterSet,
    genuineProductSet,
    canonHelperSet,
)
for d, chains in pathwaysByDepth.items():
    print(f"  {d}-step pathways found: {len(chains)}")

savedFiles = []

for depth in range(1, 4):
    chains = pathwaysByDepth[depth]
    if not chains:
        print(f"\nNo {depth}-step pathways found.")
        continue

    for pIdx, chain in enumerate(chains):
        stepPanels = []
        totalSteps = len(chain)

        for stepIdx, rxnIdx in enumerate(chain):
            isLastStep = (stepIdx == totalSteps - 1)
            stepLabel  = (
                f"{depthWords[totalSteps]} Pathway  "
                f"(Step {stepIdx + 1} of {totalSteps})"
            )
            panel = renderReactionStep(
                rxnIdx, reactionDF, canonStarterSet, genuineProductSet,
                canonHelperSet, isLastStep, stepLabel,
            )
            if panel is not None:
                stepPanels.append(panel)

        pathwayImg = stackStepPanels(stepPanels)
        if pathwayImg is None:
            continue

        baseName = f"{fileNamePrefix}_step{depth}_pathway{pIdx + 1}"
        pngPath  = f"{baseName}.png"
        pdfPath  = f"{baseName}.pdf"

        pathwayImg.save(pngPath, format="PNG", dpi=(600, 600))
        pathwayImg.save(pdfPath, format="PDF", resolution=600)
        savedFiles.append((pngPath, pdfPath))

        print(f"Saved: {pngPath}  |  {pdfPath}")
        display(pathwayImg)

print(f"\nTotal pathway figures saved: {len(savedFiles)}")

### Read `reaction_strings` from the `DORAnet` generated `json` file

In [ ]:
# Discover JSON files
allStarterDirPaths = sorted([
    p for p in glob.glob(os.path.join(doranetOutputDir, "starter_*"))
    if os.path.isdir(p)
])
print(f"Found {len(allStarterDirPaths)} starter directories")

discoveredJsonFiles        = []
directoriesWithMissingJson = []

for starterDirPath in allStarterDirPaths:
    dirName          = os.path.basename(starterDirPath)
    expectedJsonPath = os.path.join(starterDirPath, f"{dirName}_network_pretreated.json")

    if os.path.exists(expectedJsonPath):
        discoveredJsonFiles.append({
            'dirName'   : dirName,
            'dirPath'   : starterDirPath,
            'jsonPath'  : expectedJsonPath,
            'starterNum': int(dirName.replace('starter_', ''))
        })
    else:
        fallbackJsonPaths = glob.glob(os.path.join(starterDirPath, "*_network_pretreated.json"))
        if fallbackJsonPaths:
            discoveredJsonFiles.append({
                'dirName'   : dirName,
                'dirPath'   : starterDirPath,
                'jsonPath'  : fallbackJsonPaths[0],
                'starterNum': int(dirName.replace('starter_', ''))
            })
        else:
            directoriesWithMissingJson.append(dirName)

discoveredJsonFiles.sort(key=lambda x: x['starterNum'])

print(f"Found {len(discoveredJsonFiles)} JSON files")
if directoriesWithMissingJson:
    print(f"Missing JSON in {len(directoriesWithMissingJson)} directories: "
          f"{directoriesWithMissingJson[:10]}")

# Preview
for fileInfo in discoveredJsonFiles[:3]:
    print(f"  {fileInfo['dirName']} -> {os.path.basename(fileInfo['jsonPath'])}")
if len(discoveredJsonFiles) > 3:
    print(f"  ... and {len(discoveredJsonFiles) - 3} more")

# Read all JSON files

perStarterReactionRecords = []
jsonReadErrors             = []

uniqueReactantStrings   = set()
uniqueProductStrings    = set()
uniqueReactantMolecules = set()
uniqueProductMolecules  = set()

for fileInfo in discoveredJsonFiles:
    try:
        with open(fileInfo['jsonPath'], "r", encoding="utf-8") as f:
            reactionList = json.load(f)

        for rxn in reactionList:
            parts     = rxn.split(">")
            reactants = parts[0]
            ruleName  = parts[1]
            metaBlock = parts[2]
            products  = parts[3]
            plainRxn  = f"{reactants} >> {products}"

            uniqueReactantStrings.add(reactants)
            uniqueProductStrings.add(products)

            for mol in reactants.split("."):
                uniqueReactantMolecules.add(mol)
            for mol in products.split("."):
                uniqueProductMolecules.add(mol)

            perStarterReactionRecords.append({
                "reactants"            : reactants,
                "products"             : products,
                "reactionString"       : plainRxn,
                "ruleName"             : ruleName,
                "thermo"               : metaBlock.split("$")[0],
                "reactantStoich"       : metaBlock.split("$")[1],
                "productStoich"        : metaBlock.split("$")[2],
                "reactionType"         : metaBlock.split("$")[3],
                "numReactantMolecules" : len(reactants.split(".")),
                "numProductMolecules"  : len(products.split(".")),
                "SourceStarterNum"     : fileInfo['starterNum'],
                "SourceDirectory"      : fileInfo['dirName'],
            })

        #print(f"  Loaded {len(reactionList):>6} reactions from {fileInfo['dirName']}")

    except Exception as e:
        jsonReadErrors.append({
            'dirName' : fileInfo['dirName'],
            'jsonPath': fileInfo['jsonPath'],
            'error'   : str(e)
        })
        print(f"  ERROR reading {fileInfo['dirName']}: {e}")

# Step 3: Build DataFrame 

if perStarterReactionRecords:
    reactionDF = pd.DataFrame(perStarterReactionRecords).reset_index(drop=True)
else:
    raise RuntimeError("No reaction records loaded — check discoveredJsonFiles and JSON format!")

print("\n" + "-" * 55)
print(f"Total JSON files read successfully : {len(discoveredJsonFiles) - len(jsonReadErrors)}")
print(f"Total JSON files failed            : {len(jsonReadErrors)}")
print(f"Total number of reactions          : {len(reactionDF)}")
print(f"Unique reactant strings            : {len(uniqueReactantStrings)}")
print(f"Unique product strings             : {len(uniqueProductStrings)}")
print(f"Unique individual reactant SMILES  : {len(uniqueReactantMolecules)}")
print(f"Unique individual product SMILES   : {len(uniqueProductMolecules)}")
print(f"Unique source directories          : {reactionDF['SourceDirectory'].nunique()}")
print("-" * 55)

if jsonReadErrors:
    print(f"\nFailed to read {len(jsonReadErrors)} files:")
    for err in jsonReadErrors:
        print(f"  {err['dirName']}: {err['error']}")

reactionDF.to_csv("reactionDF.csv", index=False)
print("file containing all the details to run DORA_XGB is saved to reactionDF.csv")

reactionDF

### Use DORA-XGB to get feasibility score

We run this script from the command line since it contains a large number of reactions
and running it from a Jupyter cell can oversubscribe CPU cores due to the nature of
the XGBoost algorithm. Run it as:

`python runDORAXGB.py doraxgb_config.yaml`

### Keep only `high feasible` reactions

In [ ]:
reactionDF_DORAXGB = pd.read_csv("reactionDF_wDORAXGBfeasibility.csv")
reactionDF_DORAXGB

In [ ]:
feasibilityLabels = [
    "feasibilityLabel_rule1",
    "feasibilityLabel_rule2",
    "feasibilityLabel_rule3",
    "feasibilityLabel_rule4",
]
targetValue = 1

for lbl in feasibilityLabels:
    if lbl not in reactionDF_DORAXGB.columns:
        raise KeyError(f"Column '{lbl}' not found in reactionDF_DORAXGB")

results = {}
for feasibilityLabel in feasibilityLabels:
    high_df = reactionDF_DORAXGB[
        (reactionDF_DORAXGB[feasibilityLabel] == targetValue)
        & (reactionDF_DORAXGB[feasibilityLabel].notna())
    ].sort_values(by=feasibilityLabel, ascending=False).reset_index(drop=True)
    results[feasibilityLabel] = {"count_high": len(high_df), "df": high_df}

print(f"Target value considered as highly feasible: {targetValue}")
for lbl in feasibilityLabels:
    print(f"{lbl}: {results[lbl]['count_high']} highly feasible reactions")

best_rule = max(feasibilityLabels, key=lambda l: results[l]["count_high"])
bestRuleScoreCol = best_rule.replace("Label", "Score")   # needed for pathway scoring below
reactionDF_DORAXGB_highFeasibility = results[best_rule]["df"]

print(f"\nChosen feasibility label: {best_rule} == {targetValue}")
print(f"Number of high-feasibility reactions: {len(reactionDF_DORAXGB_highFeasibility)}")

In [ ]:
def canonicalizeSMILES(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None


def extractSmilesWithSource(df, columnName):
    """Like extractSmilesSet, but tracks which SourceStarterNum(s) each
    extracted SMILES actually came from, using each ROW's own SourceStarterNum.
    reactionDF_DORAXGB_highFeasibility has one row per reaction, each already
    correctly tagged with the starter directory it was read from (cell 10) --
    this preserves that instead of collapsing every molecule to one global value.
    """
    sourceMap = {}  # canonical SMILES -> set of SourceStarterNum
    for _, row in df[[columnName, "SourceStarterNum"]].dropna(subset=[columnName, "SourceStarterNum"]).iterrows():
        entry = row[columnName]
        starterNum = row["SourceStarterNum"]

        if isinstance(entry, (list, tuple)):
            rawMols = [s.strip() for s in entry if s and s.strip()]
        elif isinstance(entry, str):
            rawMols = []
            for chunk in entry.split(";"):
                chunk = chunk.strip()
                for mol in chunk.split("."):
                    mol = mol.strip()
                    if mol:
                        rawMols.append(mol)
        else:
            continue

        for rawSmi in rawMols:
            canonSmi = canonicalizeSMILES(rawSmi)
            if canonSmi is None:
                continue
            sourceMap.setdefault(canonSmi, set()).add(starterNum)

    return sourceMap


if "SourceStarterNum" not in reactionDF_DORAXGB_highFeasibility.columns:
    raise KeyError("'SourceStarterNum' not found in reactionDF_DORAXGB_highFeasibility")

# --- Extract SMILES, keeping every SourceStarterNum each one is actually
# associated with -- a molecule can legitimately be generated under more
# than one starter, and this preserves that instead of picking just one. ---
reactantSourceMap = extractSmilesWithSource(reactionDF_DORAXGB_highFeasibility, "reactants")
productSourceMap  = extractSmilesWithSource(reactionDF_DORAXGB_highFeasibility, "products")

reactantSet = set(reactantSourceMap.keys())
productSet  = set(productSourceMap.keys())
starterSet  = {c for s in user_starters if (c := canonicalizeSMILES(s)) is not None}
allMolecules = reactantSet | productSet | starterSet

# user_starters is a flat set combined across all directories (cell 8), with
# no per-job mapping of its own -- attribute each starter to whichever job(s)
# it actually shows up as a reactant in, which is its real source.
starterSourceMap = {smi: reactantSourceMap.get(smi, set()) for smi in starterSet}


def allSourceStarterNums(smiles):
    return reactantSourceMap.get(smiles, set()) | productSourceMap.get(smiles, set()) | starterSourceMap.get(smiles, set())


# --- Print summary stats ---
print(f"Unique canonical reactant SMILES : {len(reactantSet)}")
print(f"Unique canonical product  SMILES : {len(productSet)}")
print(f"Unique canonical starter  SMILES : {len(starterSet)}")
print(f"Total unique canonical    SMILES : {len(allMolecules)}")
print(f"  (reactants only)               : {len(reactantSet - productSet - starterSet)}")
print(f"  (products only)                : {len(productSet - reactantSet - starterSet)}")
print(f"  (both reactant & product)      : {len(reactantSet & productSet)}")
allStarterNumsSeen = sorted({n for smi in allMolecules for n in allSourceStarterNums(smi)})

# --- Build rows: ONE ROW PER (SMILES, SourceStarterNum) PAIR ---
# This is an explode, not a collapse: a molecule generated under multiple
# starters gets one row per starter, so groupby("SourceStarterNum") downstream
# (e.g. the PKS/RetroTide figure) sees it correctly under every starter it
# actually came from, instead of only the first one alphabetically.
rows = []
for smiles in starterSet:
    isReactant = smiles in reactantSet
    isProduct  = smiles in productSet
    for starterNum in (allSourceStarterNums(smiles) or {None}):
        rows.append({
            "SMILES"           : smiles,
            "Is_reactant"      : isReactant,
            "Is_product"       : isProduct,
            "Is_Starter"       : True,
            "SourceDirectory"  : fileNamePrefix,
            "SourceStarterNum" : starterNum,
        })

for smiles in allMolecules - starterSet:
    isReactant = smiles in reactantSet
    isProduct  = smiles in productSet
    isStarter  = isReactant and not isProduct
    for starterNum in (allSourceStarterNums(smiles) or {None}):
        rows.append({
            "SMILES"           : smiles,
            "Is_reactant"      : isReactant,
            "Is_product"       : isProduct,
            "Is_Starter"       : isStarter,
            "SourceDirectory"  : fileNamePrefix,
            "SourceStarterNum" : starterNum,
        })

reactionDF_DORAXGB_highFeasibility_uniqueMolecules = (
    pd.DataFrame(rows)
    .drop_duplicates(subset=["SMILES", "SourceStarterNum"])  # NOT "SMILES" alone -- see note below
    .reset_index(drop=True)
)

print(f"\nFinal dataframe shape: {reactionDF_DORAXGB_highFeasibility_uniqueMolecules.shape}")
print(f"Unique SMILES: {reactionDF_DORAXGB_highFeasibility_uniqueMolecules['SMILES'].nunique()}")
print(f"Unique SourceStarterNum groups: {reactionDF_DORAXGB_highFeasibility_uniqueMolecules['SourceStarterNum'].nunique()}")
reactionDF_DORAXGB_highFeasibility_uniqueMolecules

### Remove HELPERS and Deduplicate

In [ ]:
# Step 1: Remove helpers
reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers = reactionDF_DORAXGB_highFeasibility_uniqueMolecules[
    ~reactionDF_DORAXGB_highFeasibility_uniqueMolecules['SMILES'].isin(helpersToExclude)
].copy()

helpersRemovedCount = (
    len(reactionDF_DORAXGB_highFeasibility_uniqueMolecules) -
    len(reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers)
)
helpersKeptPercent = (
    len(reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers) /
    len(reactionDF_DORAXGB_highFeasibility_uniqueMolecules)
) * 100

print(f"Step 1 - Removed {helpersRemovedCount} helper molecules")
print(f"Remaining molecules: {len(reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers)} ({helpersKeptPercent:.2f}%)")

# Step 2: Already deduplicated — just confirm no duplicates exist
duplicateCount = reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers['SMILES'].duplicated().sum()
print(f"\nStep 2 - Duplicate SMILES found: {duplicateCount}")

if duplicateCount > 0:
    reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers = (
        reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers
        .drop_duplicates(subset='SMILES', keep='first')
        .reset_index(drop=True)
    )
    print(f"  Deduplicated to: {len(reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers)} molecules")
else:
    reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers = (
        reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers
        .reset_index(drop=True)
    )

# Step 4: Save
reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers.to_csv(
    f"{fileNamePrefix}_uniqueMolecules.csv", index=False, encoding="utf-8"
)
print(f"\nSaved to: {fileNamePrefix}_uniqueMolecules.csv")

reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers

In [ ]:
def countAtoms(smiles: str) -> dict:
    """Count C, O, N, S atoms in a SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {'C_count': np.nan, 'O_count': np.nan, 'N_count': np.nan, 'S_count': np.nan}

    # Add explicit hydrogens if you also want H count
    atomCounts = {'C': 0, 'O': 0, 'N': 0, 'S': 0}
    for atom in mol.GetAtoms():
        symbol = atom.GetSymbol()
        if symbol in atomCounts:
            atomCounts[symbol] += 1

    return {
        'C_count': atomCounts['C'],
        'O_count': atomCounts['O'],
        'N_count': atomCounts['N'],
        'S_count': atomCounts['S']
    }


# Apply to each SMILES
atomCountsDF = reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers['SMILES'].apply(
    lambda s: pd.Series(countAtoms(s))
)

# Concatenate with original dataframe
uniqueMolecules_molCONS = pd.concat(
    [reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers, atomCountsDF],
    axis=1
)

uniqueMolecules_molCONS

### Generate .gif file

In [ ]:
import itertools
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display, Image as IPImage

sourceDF = reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers

MOL_SIZE       = (220, 220)
ROWS_PER_FRAME = 3
GENS_PER_ROW   = 3
MAX_FRAMES     = 100

# ── Step 1: Separate molecules ────────────────────────────────────────────────

starterSmiles   = sourceDF[sourceDF['Is_Starter'] == True]['SMILES'].tolist()
generatedSmiles = sourceDF[sourceDF['Is_Starter'] == False]['SMILES'].tolist()
starterSmilesSet = set(starterSmiles)

print(f"Source directory    : {sourceDF['SourceDirectory'].unique().tolist()}")
print(f"Starter molecules   : {len(starterSmiles)}")
print(f"Generated molecules : {len(generatedSmiles)}")

# ── Step 2: Build row data ────────────────────────────────────────────────────
# Column 1 cycles through starters then generated molecules
# Columns 2-4 show consecutive chunks of 3 generated molecules

col1Cycle = itertools.cycle(starterSmiles + generatedSmiles)

rowData = []
for genStart in range(0, len(generatedSmiles), GENS_PER_ROW):
    genChunk = generatedSmiles[genStart:genStart + GENS_PER_ROW]
    rowData.append((next(col1Cycle), genChunk, genStart))

framesGrouped = [
    rowData[i:i + ROWS_PER_FRAME]
    for i in range(0, len(rowData), ROWS_PER_FRAME)
]
print(f"Total frames available : {len(framesGrouped)}")

# ── Step 3: Sample frames before rendering ────────────────────────────────────

if len(framesGrouped) > MAX_FRAMES:
    sampleIndices = np.linspace(0, len(framesGrouped) - 1, MAX_FRAMES, dtype=int)
    framesSampled = [framesGrouped[i] for i in sampleIndices]
else:
    framesSampled = framesGrouped

print(f"Frames to render       : {len(framesSampled)}")

# ── Step 4: Pre-render only column 1 images needed for sampled frames ─────────

neededCol1Smiles = {row[0] for frame in framesSampled for row in frame}
col1ImageCache   = {}

def smiles_to_image(smiles, img_size=MOL_SIZE):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        img  = Image.new('RGB', img_size, 'white')
        draw = ImageDraw.Draw(img)
        draw.text((10, img_size[1] // 2), "Invalid SMILES", fill='red')
        return img
    return Draw.MolToImage(mol, size=img_size)

for smi in neededCol1Smiles:
    col1ImageCache[smi] = smiles_to_image(smi)

print(f"Column 1 images pre-rendered : {len(col1ImageCache)}")

# ── Step 5: Frame rendering ───────────────────────────────────────────────────

def create_frame(rows, col1ImageCache, mol_size=MOL_SIZE):
    """
    Create one GIF frame with ROWS_PER_FRAME rows.
    Each row layout: [col1 molecule] --> [gen1] [gen2] [gen3]

    rows           : list of (col1_smiles, [gen_smiles, ...], start_index)
    col1ImageCache : dict of {smiles: PIL.Image} — pre-rendered column 1 images
    """
    padding      = 15
    arrow_width  = 60
    label_height = 22
    row_height   = mol_size[1] + label_height + padding * 2
    frame_height = row_height * len(rows)

    col1_x        = padding
    arrow_x_start = col1_x + mol_size[0] + padding
    arrow_x_end   = arrow_x_start + arrow_width
    gen_x_start   = arrow_x_end + padding
    gen_col_width = mol_size[0] + padding
    frame_width   = gen_x_start + GENS_PER_ROW * gen_col_width + padding

    frame = Image.new('RGB', (frame_width, frame_height), 'white')
    draw  = ImageDraw.Draw(frame)

    try:
        font       = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 12)
        title_font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 13)
    except (IOError, OSError):
        font = title_font = ImageFont.load_default()

    for row_idx, (col1_smi, gen_smiles_list, start_index) in enumerate(rows):
        mol_y   = row_idx * row_height + padding
        arrow_y = mol_y + mol_size[1] // 2

        # Column 1: pre-rendered image
        frame.paste(col1ImageCache[col1_smi], (col1_x, mol_y))
        col1_label = "Starter PKS protein" if col1_smi in starterSmilesSet else "PKS protein"
        draw.text(
            (col1_x, mol_y + mol_size[1] + 3),
            col1_label,
            fill='blue', font=title_font
        )

        # Arrow
        draw.line(
            [(arrow_x_start, arrow_y), (arrow_x_end, arrow_y)],
            fill='black', width=3
        )
        draw.polygon([
            (arrow_x_end,      arrow_y - 8),
            (arrow_x_end + 12, arrow_y),
            (arrow_x_end,      arrow_y + 8),
        ], fill='black')

        # Columns 2-4: generated molecules rendered on the fly
        for gen_idx, gen_smi in enumerate(gen_smiles_list[:GENS_PER_ROW]):
            gen_x = gen_x_start + gen_idx * gen_col_width
            frame.paste(smiles_to_image(gen_smi, mol_size), (gen_x, mol_y))
            draw.text(
                (gen_x, mol_y + mol_size[1] + 3),
                f"DORAnet Compound #{start_index + gen_idx + 1}",
                fill='green', font=font
            )

        # Row separator
        if row_idx < len(rows) - 1:
            sep_y = (row_idx + 1) * row_height
            draw.line([(0, sep_y), (frame_width, sep_y)], fill='lightgray', width=1)

    return frame

# ── Step 6: Render sampled frames ─────────────────────────────────────────────

allFrameImages = []
for frame_idx, frameRows in enumerate(framesSampled):
    allFrameImages.append(create_frame(frameRows, col1ImageCache))
    if (frame_idx + 1) % 25 == 0:
        print(f"Rendered frame {frame_idx + 1}/{len(framesSampled)}")

# ── Step 7: Save GIF ──────────────────────────────────────────────────────────

gifOutputPath = f"{fileNamePrefix}_highFeasibility.gif"
allFrameImages[0].save(
    gifOutputPath,
    save_all=True,
    append_images=allFrameImages[1:],
    duration=3000,
    loop=0
)
print(f"Saved GIF    : {gifOutputPath}")
print(f"Total frames : {len(allFrameImages)}")

display(IPImage(filename=gifOutputPath))

In [ ]:
# Ensure needed imports exist (if not already present)
from PIL import Image, ImageDraw, ImageFont
from rdkit import Chem
from rdkit.Chem import Draw
import math

basidalin_precursor_SMILES = "O=C1C=CC(=CCO)O1"
sourceDF = reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers

# -------------------------------------------------------------------
# Step 1: Pick up to 4 distinct RetroTide compounds
# -------------------------------------------------------------------
def selectStarterGroups(sourceDF, numStarters=4, minCompoundsPerStarter=2, maxCompoundsPerStarter=4):
    usedSmiles = {basidalin_precursor_SMILES}
    selectedGroups = []

    # Most-populated groups first, so genuinely richer starters get to show
    # more compounds while sparser (but still qualifying) starters show fewer.
    candidateOrder = sorted(
        sourceDF.groupby("SourceStarterNum"),
        key=lambda kv: kv[1][kv[1]["Is_Starter"] == False]["SMILES"].nunique(),
        reverse=True,
    )

    for starterNum, group in candidateOrder:
        starterRows = group[group["Is_Starter"] == True]["SMILES"].dropna().unique()
        if len(starterRows) == 0:
            continue
        retrotideSmiles = starterRows[0]
        if retrotideSmiles in usedSmiles:
            continue  # skip if this RetroTide compound duplicates one already shown

        candidateGenerated = [
            s for s in group[group["Is_Starter"] == False]["SMILES"].dropna().unique()
            if s not in usedSmiles and Chem.MolFromSmiles(s) is not None
        ]
        if len(candidateGenerated) < minCompoundsPerStarter:
            continue  # doesn't meet the minimum -- skip this starter entirely, don't pad

        chosenGenerated = candidateGenerated[:maxCompoundsPerStarter]
        usedSmiles.add(retrotideSmiles)
        usedSmiles.update(chosenGenerated)

        selectedGroups.append({
            "starterNum": starterNum,
            "retrotideSmiles": retrotideSmiles,
            "generatedSmiles": chosenGenerated,
        })
        if len(selectedGroups) == numStarters:
            break

    return selectedGroups


selectedGroups = selectStarterGroups(sourceDF, numStarters=4, minCompoundsPerStarter=2, maxCompoundsPerStarter=3)
print(f"Selected {len(selectedGroups)} RetroTide starter groups:")
for g in selectedGroups:
    print(f"  Starter #{g['starterNum']}: {len(g['generatedSmiles'])} generated compounds")
    print(f"    RetroTide: {g['retrotideSmiles']}")
    for s in g["generatedSmiles"]:
        print(f"      -> {s}")

# No crash: proceed with however many groups qualified, even if fewer than 4.
if not selectedGroups:
    print("\nWARNING: no starter groups met minCompoundsPerStarter "
          f"({2}). Nothing to plot -- check sourceDF or lower the minimum.")
elif len(selectedGroups) < 4:
    print(f"\nNote: only {len(selectedGroups)} qualifying groups found (requested 4). "
          "Proceeding with what's available rather than padding or crashing.")


# -------------------------------------------------------------------
# Step 2: Draw the branching figure -- PKS starter, arrows to each RetroTide
# compound, each followed by its own generated compounds (variable count per row).
# -------------------------------------------------------------------
def smiles_to_image(smiles, img_size=(220, 220)):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        img = Image.new('RGB', img_size, 'white')
        ImageDraw.Draw(img).text((10, img_size[1] // 2), "Invalid SMILES", fill='red')
        return img
    return Draw.MolToImage(mol, size=img_size)


def draw_arrow(draw, x1, y1, x2, y2, color='black', width=3, head_len=12, head_angle_deg=20):
    draw.line([(x1, y1), (x2, y2)], fill=color, width=width)
    angle = math.atan2(y2 - y1, x2 - x1)
    head_angle = math.radians(head_angle_deg)
    lx = x2 - head_len * math.cos(angle - head_angle)
    ly = y2 - head_len * math.sin(angle - head_angle)
    rx = x2 - head_len * math.cos(angle + head_angle)
    ry = y2 - head_len * math.sin(angle + head_angle)
    draw.polygon([(x2, y2), (lx, ly), (rx, ry)], fill=color)


def create_pks_retrotide_doranet_figure(pks_starter_smi, selectedGroups, mol_size=(220, 220)):
    padding, gap, label_height = 20, 30, 40
    mol_w, mol_h = mol_size
    numRows = len(selectedGroups)

    # Grid width is set by whichever row has the MOST generated compounds --
    # rows with fewer simply leave the remaining columns blank, not padded.
    # (1 + maxGeneratedCols: the +1 is the RetroTide column itself, separate
    # from the generated-compound columns.)
    maxGeneratedCols = max(len(g["generatedSmiles"]) for g in selectedGroups)
    totalDataCols = 1 + maxGeneratedCols

    row_h = mol_h + label_height
    grid_height = numRows * row_h + (numRows - 1) * gap

    frame_width = padding * 3 + mol_w + totalDataCols * mol_w + totalDataCols * gap
    frame_height = padding * 2 + grid_height

    frame = Image.new('RGB', (frame_width, frame_height), 'white')
    draw = ImageDraw.Draw(frame)

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14)
        title_font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 15)
    except (IOError, OSError):
        font = ImageFont.load_default()
        title_font = ImageFont.load_default()

    starter_x = padding
    starter_y = padding + (grid_height - mol_h) // 2
    starter_img = smiles_to_image(pks_starter_smi, mol_size)
    frame.paste(starter_img, (starter_x, starter_y))
    draw.text((starter_x, starter_y + mol_h + 4), "PKS compound (Starter)", fill='blue', font=title_font)

    starter_right_x = starter_x + mol_w
    starter_center_y = starter_y + mol_h // 2
    col_x = [starter_right_x + padding * 2 + c * (mol_w + gap) for c in range(totalDataCols)]

    for rowIdx, group in enumerate(selectedGroups):
        row_y = padding + rowIdx * (row_h + gap)

        # Column 0 of the data area: RetroTide-diversified compound for this row
        retro_img = smiles_to_image(group["retrotideSmiles"], mol_size)
        frame.paste(retro_img, (col_x[0], row_y))
        draw.text((col_x[0] +50, row_y + mol_h + 4), f"PKS Compound", fill='purple', font=font)

        retro_center_y = row_y + mol_h // 2
        draw_arrow(draw, starter_right_x + 5, starter_center_y, col_x[0] - 5, retro_center_y,
                   color='black', width=3, head_len=12, head_angle_deg=20)

        # Remaining columns: as many DORAnet compounds as THIS row has --
        # shorter rows just leave the rest of the row blank.
        for genIdx, genSmi in enumerate(group["generatedSmiles"]):
            gen_x = col_x[genIdx + 1]
            gen_img = smiles_to_image(genSmi, mol_size)
            frame.paste(gen_img, (gen_x, row_y))
            draw.text((gen_x + 50, row_y + mol_h + 4), f"Generated Compound", fill='green', font=font)

    return frame


if selectedGroups:
    publicationFigure = create_pks_retrotide_doranet_figure(basidalin_precursor_SMILES, selectedGroups, mol_size=(220, 220))

    # ---------------------------------------------------------------
    # Step 3: Save as PNG and PDF for the manuscript
    # ---------------------------------------------------------------
    pngOutputPath = f"{fileNamePrefix}_PKS_RetroTide_DORAnet_compounds.png"
    pdfOutputPath = f"{fileNamePrefix}_PKS_RetroTide_DORAnet_compounds.pdf"

    publicationFigure.save(pngOutputPath, dpi=(300, 300))
    publicationFigure.save(pdfOutputPath, "PDF", resolution=300.0)

    print(f"Saved PNG: {pngOutputPath}")
    print(f"Saved PDF: {pdfOutputPath}")
    print(f"Figure size: {publicationFigure.width} x {publicationFigure.height} px")

    from IPython.display import display
    display(publicationFigure)

### Drug-Likeness Analysis of molecules present in `uniqueMolecules`

Drug-likeness is a qualitative assessment used in drug discovery to determine whether a molecule has physicochemical and structural properties consistent with orally active drugs. Molecules that satisfy drug-likeness criteria are more likely to exhibit favorable **absorption, distribution, metabolism, excretion, and toxicity (ADMET)** profiles.

In this analysis, each unique molecule is evaluated against **five widely used drug-likeness filters**:

### 1. Lipinski's Rule of Five (Ro5)
Proposed by **Christopher Lipinski (1997)**, this is the most well-known drug-likeness filter. It predicts poor absorption or permeation when **more than one** of the following conditions is violated:

- Molecular Weight ≤ 500 Da
- LogP ≤ 5
- Hydrogen Bond Donors (HBD) ≤ 5
- Hydrogen Bond Acceptors (HBA) ≤ 10

> **Reference:** Lipinski, C.A. et al. *Adv. Drug Deliv. Rev.* **23**, 3–25 (1997). [DOI:10.1016/S0169-409X(96)00423-1](https://doi.org/10.1016/S0169-409X(96)00423-1)

### 2. Veber's Rules
Proposed by **Veber et al. (2002)**, these rules focus on oral bioavailability based on molecular flexibility and polar surface area:

- Number of Rotatable Bonds ≤ 10
- Topological Polar Surface Area (TPSA) ≤ 140 Å²

> **Reference:** Veber, D.F. et al. *J. Med. Chem.* **45**, 2615–2623 (2002). [DOI:10.1021/jm020017n](https://doi.org/10.1021/jm020017n)

### 3. Ghose Filter
Proposed by **Ghose et al. (1999)**, this filter defines a drug-like chemical space using ranges for key properties:

- 160 ≤ Molecular Weight ≤ 480 Da
- −0.4 ≤ LogP ≤ 5.6
- 20 ≤ Number of Heavy Atoms ≤ 70
- 40 ≤ Molar Refractivity ≤ 130

> **Reference:** Ghose, A.K. et al. *J. Comb. Chem.* **1**, 55–68 (1999). [DOI:10.1021/cc9800071](https://doi.org/10.1021/cc9800071)

### 4. Egan Filter
Proposed by **Egan et al. (2000)**, this filter predicts passive intestinal absorption using a simple two-parameter model:

- LogP ≤ 5.88
- TPSA ≤ 131.6 Å²

> **Reference:** Egan, W.J. et al. *J. Med. Chem.* **43**, 3867–3877 (2000). [DOI:10.1021/jm000292e](https://doi.org/10.1021/jm000292e)

### 5. Muegge Filter
Proposed by **Muegge et al. (2001)**, this pharmacophore-based filter uses a broader set of criteria to identify drug-like molecules:

- 200 ≤ Molecular Weight ≤ 600 Da
- −2 ≤ LogP ≤ 5
- TPSA ≤ 150 Å²
- Number of Rings ≤ 7
- Hydrogen Bond Acceptors ≤ 10
- Hydrogen Bond Donors ≤ 5
- Number of Rotatable Bonds ≤ 15
- Number of Heavy Atoms ≥ 8

> **Reference:** Muegge, I. et al. *J. Med. Chem.* **44**, 1841–1846 (2001). [DOI:10.1021/jm015507e](https://doi.org/10.1021/jm015507e)

### Drug-Like Score

Each molecule receives a **DrugLikeScore** ranging from **0 to 5**, representing the number of filters it passes. A score of **5** indicates the molecule satisfies all five drug-likeness criteria, suggesting strong potential as an orally bioavailable drug candidate.

| DrugLikeScore | Interpretation |
|:---:|:---|
| 5 | Excellent drug-likeness — passes all filters |
| 4 | Strong drug-likeness — minor deviation in one filter |
| 3 | Moderate drug-likeness |
| 2 | Weak drug-likeness |
| 1 | Poor drug-likeness |
| 0 | Not drug-like by any standard filter |

### Molecular Properties Computed

| Property | Description | Used By |
|:---|:---|:---|
| Molecular Weight (MW) | Exact molecular weight in Daltons | Lipinski, Ghose, Muegge |
| LogP | Octanol-water partition coefficient (lipophilicity) | Lipinski, Ghose, Egan, Muegge |
| HBD | Number of hydrogen bond donors | Lipinski, Muegge |
| HBA | Number of hydrogen bond acceptors | Lipinski, Muegge |
| TPSA | Topological polar surface area (Å²) | Veber, Egan, Muegge |
| Rotatable Bonds | Number of rotatable bonds | Veber, Muegge |
| Heavy Atoms | Number of non-hydrogen atoms | Ghose, Muegge |
| Molar Refractivity | Measure of molecular polarizability | Ghose |
| Rings | Total number of rings | Muegge |
| Aromatic Rings | Number of aromatic rings | Structural characterization |

> **Note:** These filters are designed primarily for **orally active small molecules**. Natural products and biologics may legitimately violate some criteria while still being therapeutically relevant. The DrugLikeScore should be interpreted as a guide, not an absolute cutoff.

In [ ]:
from rdkit.Chem import Descriptors, rdMolDescriptors, Lipinski


# Drug-likeness filter functions
def computeDrugLikeProperties(smiles):
    """Compute drug-likeness properties from SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return pd.Series({
            'MolWeight': None,
            'LogP': None,
            'NumHBD': None,
            'NumHBA': None,
            'TPSA': None,
            'NumRotatableBonds': None,
            'NumRings': None,
            'NumAromaticRings': None,
            'NumHeavyAtoms': None,
            'MolarRefractivity': None,
            'PassesLipinski': False,
            'LipinskiViolations': None,
            'PassesVeber': False,
            'PassesGhose': False,
            'PassesEgan': False,
            'PassesMuegge': False,
            'DrugLikeScore': 0,
            'IsValidMol': False
        })

    molWeight = Descriptors.ExactMolWt(mol)
    logP = Descriptors.MolLogP(mol)
    numHBD = rdMolDescriptors.CalcNumHBD(mol)
    numHBA = rdMolDescriptors.CalcNumHBA(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    numRotatableBonds = rdMolDescriptors.CalcNumRotatableBonds(mol)
    numRings = rdMolDescriptors.CalcNumRings(mol)
    numAromaticRings = rdMolDescriptors.CalcNumAromaticRings(mol)
    numHeavyAtoms = mol.GetNumHeavyAtoms()
    molarRefractivity = Descriptors.MolMR(mol)

    lipinskiViolations = 0
    if molWeight > 500:
        lipinskiViolations += 1
    if logP > 5:
        lipinskiViolations += 1
    if numHBD > 5:
        lipinskiViolations += 1
    if numHBA > 10:
        lipinskiViolations += 1
    passesLipinski = lipinskiViolations <= 1

    passesVeber = (numRotatableBonds <= 10) and (tpsa <= 140)

    passesGhose = (
        (160 <= molWeight <= 480) and
        (-0.4 <= logP <= 5.6) and
        (20 <= numHeavyAtoms <= 70) and
        (40 <= molarRefractivity <= 130)
    )

    passesEgan = (logP <= 5.88) and (tpsa <= 131.6)

    passesMuegge = (
        (200 <= molWeight <= 600) and
        (-2 <= logP <= 5) and
        (tpsa <= 150) and
        (numRings <= 7) and
        (numHBA <= 10) and
        (numHBD <= 5) and
        (numRotatableBonds <= 15) and
        (numHeavyAtoms >= 8)
    )

    drugLikeScore = sum([
        passesLipinski,
        passesVeber,
        passesGhose,
        passesEgan,
        passesMuegge
    ])

    return pd.Series({
        'MolWeight': round(molWeight, 4),
        'LogP': round(logP, 4),
        'NumHBD': numHBD,
        'NumHBA': numHBA,
        'TPSA': round(tpsa, 2),
        'NumRotatableBonds': numRotatableBonds,
        'NumRings': numRings,
        'NumAromaticRings': numAromaticRings,
        'NumHeavyAtoms': numHeavyAtoms,
        'MolarRefractivity': round(molarRefractivity, 4),
        'PassesLipinski': passesLipinski,
        'LipinskiViolations': lipinskiViolations,
        'PassesVeber': passesVeber,
        'PassesGhose': passesGhose,
        'PassesEgan': passesEgan,
        'PassesMuegge': passesMuegge,
        'DrugLikeScore': drugLikeScore,
        'IsValidMol': True
    })


# Compute drug-likeness for all molecules
print(f"Computing drug-likeness properties for {len(uniqueMolecules)} molecules...")

drugLikeProperties = uniqueMolecules['SMILES'].apply(computeDrugLikeProperties)

uniqueMoleculesDrug = pd.concat(
    [uniqueMolecules.reset_index(drop=True), drugLikeProperties],
    axis=1
)

invalidMolCount = len(uniqueMoleculesDrug[~uniqueMoleculesDrug['IsValidMol']])
print(f"Invalid molecules: {invalidMolCount}")
print(f"Final DataFrame shape: {uniqueMoleculesDrug.shape}")

# Summary
validMols = uniqueMoleculesDrug[uniqueMoleculesDrug['IsValidMol']]
totalValid = len(validMols)

lipinskiCount = validMols['PassesLipinski'].sum()
veberCount = validMols['PassesVeber'].sum()
ghoseCount = validMols['PassesGhose'].sum()
eganCount = validMols['PassesEgan'].sum()
mueggeCount = validMols['PassesMuegge'].sum()
allFiveCount = len(validMols[validMols['DrugLikeScore'] == 5])
nonePassedCount = len(validMols[validMols['DrugLikeScore'] == 0])

print("\nDrug likeliness summary:")
print(f"Total valid molecules: {totalValid}")
print(f"  Passes Lipinski (Ro5):  {lipinskiCount} ({lipinskiCount/totalValid*100:.2f}%)")
print(f"  Passes Veber:           {veberCount} ({veberCount/totalValid*100:.2f}%)")
print(f"  Passes Ghose:           {ghoseCount} ({ghoseCount/totalValid*100:.2f}%)")
print(f"  Passes Egan:            {eganCount} ({eganCount/totalValid*100:.2f}%)")
print(f"  Passes Muegge:          {mueggeCount} ({mueggeCount/totalValid*100:.2f}%)")
print(f"\n  Passes ALL 5 filters:   {allFiveCount} ({allFiveCount/totalValid*100:.2f}%)")
print(f"  Passes NONE:            {nonePassedCount} ({nonePassedCount/totalValid*100:.2f}%)")

# DrugLikeScore distribution
print(f"\nDrugLikeScore Distribution:")
for score in range(6):
    count = len(validMols[validMols['DrugLikeScore'] == score])
    print(f"  Score {score}/5: {count} ({count/totalValid*100:.2f}%)")

# Breakdown: starters vs generated
startersDrug = validMols[validMols['Is_Starter'] == True]
generatedDrug = validMols[validMols['Is_Starter'] == False]

starterAllFive = len(startersDrug[startersDrug['DrugLikeScore'] == 5])
generatedAllFive = len(generatedDrug[generatedDrug['DrugLikeScore'] == 5])

starterPercent = (starterAllFive / len(startersDrug) * 100) if len(startersDrug) > 0 else 0
generatedPercent = (generatedAllFive / len(generatedDrug) * 100) if len(generatedDrug) > 0 else 0

print(f"\nAmong starters:  {starterAllFive}/{len(startersDrug)} pass all 5 ({starterPercent:.2f}%)")
print(f"Among generated: {generatedAllFive}/{len(generatedDrug)} pass all 5 ({generatedPercent:.2f}%)")

uniqueMoleculesDrug

### Check matches for Natural Product from public data bases via `SMILES`
 - NPAtlas: https://www.npatlas.org/download
 - COCONUT: https://coconut.naturalproducts.net/download

## Natural Product Matching via `InChIKey`

# How many pathways has been generated

In [ ]:
import PyPDF2

# Automatically find all starter directories (exclude files)
allStarterDirPaths = sorted([
    p for p in glob.glob(os.path.join(doranetOutputDir, "starter_*"))
    if os.path.isdir(p)
])

totalStarterDirs = len(allStarterDirPaths)
print(f"Found {totalStarterDirs} starter molecules")

# Count pathways from PDF files in each directory
pathwayCountsPerStarter = []

for starterDirPath in allStarterDirPaths:
    dirName = os.path.basename(starterDirPath)
    starterNum = int(dirName.replace('starter_', ''))

    expectedPdfPath = os.path.join(starterDirPath, f"{dirName}_pathways_visualized.pdf")

    if not os.path.exists(expectedPdfPath):
        fallbackPdfPaths = glob.glob(os.path.join(starterDirPath, "*_pathways_visualized.pdf"))
        expectedPdfPath = fallbackPdfPaths[0] if fallbackPdfPaths else None

    numPathways = 0
    if expectedPdfPath and os.path.exists(expectedPdfPath):
        try:
            with open(expectedPdfPath, 'rb') as pdfFile:
                pdfReader = PyPDF2.PdfReader(pdfFile)
                numPathways = len(pdfReader.pages)
        except Exception as e:
            print(f"Error reading PDF for {dirName}: {e}")

    pathwayCountsPerStarter.append({
        'StarterDirectory': dirName,
        'StarterNum': starterNum,
        'NumPathways': numPathways
    })

allPathwaysRaw = pd.DataFrame(pathwayCountsPerStarter).sort_values('StarterNum').reset_index(drop=True)

# Keep only starters with pathways
allPathways = allPathwaysRaw[allPathwaysRaw['NumPathways'] > 0].copy().reset_index(drop=True)

totalPathways = allPathways['NumPathways'].sum()
startersWithPathways = len(allPathways)
startersWithoutPathways = totalStarterDirs - startersWithPathways
withPercent = (startersWithPathways / totalStarterDirs) * 100
withoutPercent = (startersWithoutPathways / totalStarterDirs) * 100

print(f"Total pathways found: {totalPathways}")
print(f"Starters with pathways: {startersWithPathways} ({withPercent:.2f}%), "
      f"without: {startersWithoutPathways} ({withoutPercent:.2f}%)")

allPathways = allPathways.drop(columns=['StarterNum'])
allPathways

### Extract pathway infromation to products and reactants 

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import glob


# Helper Functions
def readTextFile(txtPath: str) -> str:
    """Read a text file and return its content."""
    with open(txtPath, "r", encoding="utf-8") as f:
        return f.read()


def splitReactionSmiles(rxnSmiles: str):
    """Split a reaction SMILES into reactants and products."""
    left, right = rxnSmiles.split(">>")
    reactants = [s for s in left.split(".") if s]
    products = [s for s in right.split(".") if s]
    return reactants, products

def getUniqueCanonicalSmiles(column: pd.Series) -> set:
    """Extract unique canonical SMILES from a pipe-separated column."""
    allSmiles = set()
    for entry in column.dropna():
        # Split by ' | ' (step separator) then by '.' (molecule separator)
        for step in entry.split(' | '):
            for smi in step.split('.'):
                smi = smi.strip()
                if smi:
                    mol = Chem.MolFromSmiles(smi)
                    if mol is not None:
                        canonical = Chem.MolToSmiles(mol)
                        allSmiles.add(canonical)
    return allSmiles

# Parser
def parseRankedPathwayTxtForDF(txt: str) -> list:
    """Parse ranked pathway text and extract structured data for each pathway."""
    blocks = re.split(r"\branking\s+", txt.strip())
    pathwayRecords = []

    for block in blocks:
        if not block.strip():
            continue

        # Extract ranking number
        rankMatch = re.match(r"(\d+)\s*", block)
        if not rankMatch:
            continue
        rank = int(rankMatch.group(1))

        lines = block.splitlines()

        # Extract scores
        def findFloat(pattern, default=float("nan")):
            m = re.search(pattern, block)
            return float(m.group(1)) if m else default

        finalScore = findFloat(r"final\s+score\s+([0-9.]+)")
        atomicEconomy = findFloat(r"atomic\s+economy\s+([0-9.]+)")

        # Extract pathway by-product
        pathwayByProduct = findFloat(r"pathway\s+by-product\s+([0-9.]+)")

        # Extract reaction SMILES (lines containing '>>' but not headers)
        reactionLines = []
        for line in lines:
            line = line.strip()
            if ">>" in line and not line.lower().startswith("reaction smiles"):
                reactionLines.append(line)

        # Extract rule names
        ruleNames = []
        for line in lines:
            line = line.strip()
            if line.startswith("rule"):
                ruleNames.append(line)

        # Extract thermodynamic data after rule lines
        # In your file format, thermo entries follow rule lines (one per step)
        # They can be "No_Thermo" or actual numeric values
        thermoValues = []
        ruleIndices = [i for i, ln in enumerate(lines) if lines[i].strip().startswith("rule")]

        if ruleIndices:
            lastRuleIdx = ruleIndices[-1]
            for line in lines[lastRuleIdx + 1:]:
                line = line.strip()
                if not line:
                    continue
                if line == "No_Thermo":
                    thermoValues.append(float("nan"))
                else:
                    try:
                        thermoValues.append(float(line))
                    except ValueError:
                        continue

        # Extract stoichiometry
        stoichiometry = ""
        for line in lines:
            if "stoichiometry" in line.lower():
                stoichiometry = line.strip()
                break

        # Extract reactants and products per step
        perStepReactants = []
        perStepProducts = []
        for rxnSmiles in reactionLines:
            reactants, products = splitReactionSmiles(rxnSmiles)
            perStepReactants.append('.'.join(reactants))
            perStepProducts.append('.'.join(products))

        # Format thermo per step
        thermoPerStep = []
        for i in range(len(reactionLines)):
            if i < len(thermoValues):
                val = thermoValues[i]
                thermoPerStep.append("No_Thermo" if pd.isna(val) else f"{val:.4f}")
            else:
                thermoPerStep.append("N/A")

        # Check if any thermo data is available
        hasThermo = any(not pd.isna(v) for v in thermoValues) if thermoValues else False

        # Compute average Gibbs free energy (only from available values)
        validThermoValues = [v for v in thermoValues if not pd.isna(v)]
        avgGibbsFreeEnergy = (
            sum(validThermoValues) / len(validThermoValues)
            if validThermoValues else float("nan")
        )

        pathwayRecords.append({
            'PathwayNumber': rank,
            'FinalScore': finalScore,
            'AtomicEconomy': atomicEconomy,
            'PathwayByProduct': pathwayByProduct,
            'NumSteps': len(reactionLines),
            'Reactant': ' | '.join(perStepReactants),
            'Product': ' | '.join(perStepProducts),
            'ReactionSMILES': ' | '.join(reactionLines),
            'RuleNames': ' | '.join(ruleNames),
            'Stoichiometry': stoichiometry,
            'ThermoPerStep': ' | '.join(thermoPerStep),
            'HasThermo': hasThermo,
            'AvgGibbsFreeEnergy': avgGibbsFreeEnergy
        })

    return pathwayRecords



# DataFrame Builder
def buildPathwaysDF(doranetOutputDir: str) -> pd.DataFrame:
    """
    Automatically find all *_ranked_pathways.txt files inside
    starter_* directories under doranetOutputDir, parse them,
    and build a consolidated DataFrame.
    """
    allRecords = []

    # Automatically find all ranked pathway text files
    txtFilePattern = os.path.join(doranetOutputDir, "starter_*", "*_ranked_pathways.txt")
    txtFiles = sorted(glob.glob(txtFilePattern))

    totalStarterDirs = len(sorted(glob.glob(os.path.join(doranetOutputDir, "starter_*"))))

    print(f"Total starter directories: {totalStarterDirs}")
    print(f"Found {len(txtFiles)} ranked pathway text files")

    parsedCount = 0
    errorCount = 0
    emptyCount = 0

    for txtPath in txtFiles:
        # Extract starter directory name from file path
        dirName = os.path.basename(os.path.dirname(txtPath))

        try:
            txt = readTextFile(txtPath)

            # Skip empty files
            if not txt.strip():
                emptyCount += 1
                continue

            pathwayRecords = parseRankedPathwayTxtForDF(txt)

            if pathwayRecords:
                for record in pathwayRecords:
                    record['StarterDirectory'] = dirName
                    allRecords.append(record)
                parsedCount += 1
            else:
                emptyCount += 1

        except Exception as e:
            print(f"  Error parsing {dirName}: {e}")
            errorCount += 1

    print(f"\nSuccessfully parsed: {parsedCount} starters")
    print(f"Empty or no pathways: {emptyCount} starters")
    if errorCount > 0:
        print(f"Errors: {errorCount} starters")
    print(f"Starters without pathway file: {totalStarterDirs - len(txtFiles)}")

    # Create DataFrame
    pathwaysDF = pd.DataFrame(allRecords)

    if not pathwaysDF.empty:
        # Count pathways per starter
        numPathwaysPerStarter = (
            pathwaysDF.groupby('StarterDirectory')['PathwayNumber']
            .count()
            .reset_index()
            .rename(columns={'PathwayNumber': 'NumPathways'})
        )
        pathwaysDF = pathwaysDF.merge(numPathwaysPerStarter, on='StarterDirectory', how='left')

        # Reorder columns
        columnOrder = [
            'StarterDirectory', 'NumPathways', 'PathwayNumber',
            'FinalScore', 'AtomicEconomy', 'PathwayByProduct',
            'NumSteps', 'Reactant', 'Product',
            'ReactionSMILES', 'RuleNames', 'Stoichiometry',
            'ThermoPerStep', 'HasThermo', 'AvgGibbsFreeEnergy'
        ]
        existingColumns = [c for c in columnOrder if c in pathwaysDF.columns]
        pathwaysDF = pathwaysDF[existingColumns]

    return pathwaysDF


# Get pathways
pathwaysDF = buildPathwaysDF(doranetOutputDir)

print(f"\nTotal pathways: {len(pathwaysDF)}")

if not pathwaysDF.empty:
    print(f"Total starters with pathways: {pathwaysDF['StarterDirectory'].nunique()}")
    #print(f"Columns: {pathwaysDF.columns.tolist()}")

    # Thermo summary
    thermoCount = pathwaysDF['HasThermo'].sum()
    noThermoCount = (~pathwaysDF['HasThermo']).sum()
    print(f"\nPathways with thermodynamic data: {thermoCount}")
    print(f"Pathways without thermodynamic data: {noThermoCount}")

    if thermoCount > 0:
        print("\nSample pathways with thermodynamic data:")
        print(pathwaysDF[pathwaysDF['HasThermo']][
            ['StarterDirectory', 'PathwayNumber', 'ThermoPerStep', 'AvgGibbsFreeEnergy']
        ].head(10).to_string(index=False))

    
else:
    print("No pathways found.")
    print(f"Check that txt files exist at: {doranetOutputDir}/starter_*/{{name}}_ranked_pathways.txt")


# Count unique canonical products and reactants
uniqueReactants = getUniqueCanonicalSmiles(pathwaysDF['Reactant'])
uniqueProducts = getUniqueCanonicalSmiles(pathwaysDF['Product'])

print(f"\nUnique canonical reactants: {len(uniqueReactants)}")
print(f"Unique canonical products: {len(uniqueProducts)}")

# Molecules that appear as both reactant and product (intermediates)
intermediates = uniqueReactants & uniqueProducts
print(f"Shared (intermediates): {len(intermediates)}")

# Molecules unique to reactants or products
onlyReactants = uniqueReactants - uniqueProducts
onlyProducts = uniqueProducts - uniqueReactants
print(f"Only in reactants (starting materials): {len(onlyReactants)}")
print(f"Only in products (final products + byproducts): {len(onlyProducts)}")

pathwaysDF = pathwaysDF.drop(columns=['FinalScore', 'PathwayByProduct', 'NumSteps', 'RuleNames', 'Stoichiometry', 'ThermoPerStep', 'HasThermo', 'AvgGibbsFreeEnergy'])
pathwaysDF.to_csv(os.path.join(doranetOutputDir, f"{fileNamePrefix}_pathways.csv"), index=False, encoding="utf-8")
pathwaysDF.head()

### Visualize pathways

### Discover All Starter Directories

In [ ]:
doranetOutputDir_NP = "doranet_output_NP"

helpersToExclude = {
    'O', 'O=O', '[H][H]', 'O=C=O', 'C=O', '[C-]#[O+]', 'Br', '[Br][Br]',
    'CO', 'C=C', 'O=S(O)O', 'N', 'O=S(=O)(O)O', 'O=NO', 'N#N',
    'O=[N+]([O-])O', 'NO', 'C#N', 'S', 'O=S=O', 'N#CO', '[H+]', 'OO',
    'Cl', 'I', 'O=C(O)O', 'O=P(O)(O)O', 'O=P(O)(O)OP(=O)(O)O', 'C',
    'CC', 'CC=O', 'CC(=O)O', 'CCC(=O)O'
}

print(f"Output directory: {doranetOutputDir_NP}")
print(f"Number of helpers to exclude: {len(helpersToExclude)}")

In [ ]:
# Automatically find all starter directories
allStarterDirPaths_NP = sorted(glob.glob(os.path.join(doranetOutputDir_NP, "starter_*_NP")))

print(f"Found {len(allStarterDirPaths_NP)} starter directories")

# Find the molecules CSV file inside each directory
discoveredCsvFiles = []
directoriesWithMissingCsv = []

for starterDirPath in allStarterDirPaths_NP:
    dirName = os.path.basename(starterDirPath)

    # Expected CSV file name matches directory name
    expectedCsvPath = os.path.join(starterDirPath, f"{dirName}_molecules.csv")

    if os.path.exists(expectedCsvPath):
        discoveredCsvFiles.append({
            'dirName': dirName,
            'dirPath': starterDirPath,
            'csvPath': expectedCsvPath,
            'starterNum': int(dirName.replace('starter_', '').replace('_NP', ''))
        })
    else:
        # Try to find any molecules CSV in the directory as fallback
        fallbackCsvPaths = glob.glob(os.path.join(starterDirPath, "*_molecules.csv"))
        if fallbackCsvPaths:
            discoveredCsvFiles.append({
                'dirName': dirName,
                'dirPath': starterDirPath,
                'csvPath': fallbackCsvPaths[0],
                'starterNum': int(dirName.replace('starter_', '').replace('_NP', ''))
            })
        else:
            directoriesWithMissingCsv.append(dirName)

# Sort by starter number
discoveredCsvFiles.sort(key=lambda x: x['starterNum'])

print(f"Found {len(discoveredCsvFiles)} CSV files containing DORAnet generated molecules")
if directoriesWithMissingCsv:
    print(f"Missing CSV in {len(directoriesWithMissingCsv)} directories: "
          f"{directoriesWithMissingCsv[:10]}...")

### Read All CSV Files into a DataFrame

In [ ]:
# Read all CSV files and combine into one DataFrame
perStarterDataFrames = []
csvReadErrors = []

for fileInfo in discoveredCsvFiles:
    try:
        singleStarterMolecules = pd.read_csv(fileInfo['csvPath'])

        # Add source starter information
        singleStarterMolecules['SourceStarterNum'] = fileInfo['starterNum']
        singleStarterMolecules['SourceDirectory'] = fileInfo['dirName']

        perStarterDataFrames.append(singleStarterMolecules)

    except Exception as e:
        csvReadErrors.append({
            'dirName': fileInfo['dirName'],
            'csvPath': fileInfo['csvPath'],
            'error': str(e)
        })

# Concatenate all DataFrames
if perStarterDataFrames:
    allMoleculesWithDuplicates_wNP = pd.concat(perStarterDataFrames, ignore_index=True)
    print(f"Total rows read (with duplicates and helpers): {len(allMoleculesWithDuplicates_wNP)}")
    print(f"Columns: {list(allMoleculesWithDuplicates_wNP.columns)}")
    print(f"Unique SMILES (before cleaning): {allMoleculesWithDuplicates_wNP['SMILES'].nunique()}")
else:
    print("ERROR: No CSV files could be read!")

if csvReadErrors:
    print(f"\nFailed to read {len(csvReadErrors)} files:")
    for err in csvReadErrors[:5]:
        print(f"  {err['dirName']}: {err['error']}")

allMoleculesWithDuplicates_wNP

### Remove HELPERS and Deduplicate

In [ ]:
# Step 1: Remove helpers
reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers = allMoleculesWithDuplicates_wNP[
    ~allMoleculesWithDuplicates_wNP['SMILES'].isin(helpersToExclude)
].copy()

helpersRemovedCount = len(allMoleculesWithDuplicates_wNP) - len(reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers)
helpersKeptPercent = (len(reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers) / len(allMoleculesWithDuplicates_wNP)) * 100
print(f"Step 1 - Removed {helpersRemovedCount} helper molecules")
print(f"Remaining molecules: {len(reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers)} ({helpersKeptPercent:.2f}%)")

# Step 2: Aggregate source starters per unique SMILES
sourceStarterMapping = (
    reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers
    .groupby('SMILES')['SourceStarterNum']
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)
sourceStarterMapping.columns = ['SMILES', 'SourceStarterList']
sourceStarterMapping['NumSourceStarters'] = sourceStarterMapping['SourceStarterList'].apply(len)
sourceStarterMapping['SourceStarters'] = sourceStarterMapping['SourceStarterList'].apply(
    lambda x: ';'.join(map(str, x))
)

# Also aggregate source directories per unique SMILES
sourceDirectoryMapping = (
    reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers
    .groupby('SMILES')['SourceDirectory']
    .apply(lambda x: ';'.join(sorted(set(x))))
    .reset_index()
)
sourceDirectoryMapping.columns = ['SMILES', 'SourceDirectories']

# Step 3: Deduplicate - keep first occurrence of each SMILES
uniqueMolecules_wNP = reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers.drop_duplicates(
    subset='SMILES', keep='first'
).copy()

# Keep only desired columns
uniqueMolecules_wNP = uniqueMolecules_wNP[['SMILES', 'Is_Starter']].copy()

# Step 4: Merge aggregated source starter and directory information
uniqueMolecules_wNP = uniqueMolecules_wNP.merge(
    sourceStarterMapping[['SMILES', 'NumSourceStarters', 'SourceStarters']],
    on='SMILES',
    how='left'
)
uniqueMolecules_wNP = uniqueMolecules_wNP.merge(
    sourceDirectoryMapping,
    on='SMILES',
    how='left'
)

duplicatesRemovedCount = len(reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers) - len(uniqueMolecules_wNP)
dedupeKeptPercent = (len(uniqueMolecules_wNP) / len(reactionDF_DORAXGB_highFeasibility_uniqueMolecules_noHelpers)) * 100
overallKeptPercent = (len(uniqueMolecules_wNP) / len(allMoleculesWithDuplicates_wNP)) * 100

print(f"Step 2 - Removed {duplicatesRemovedCount} duplicate molecular SMILES ({dedupeKeptPercent:.2f}%)")
print(f"Final unique molecules: {len(uniqueMolecules_wNP)} ({overallKeptPercent:.2f}% of original)")

uniqueMolecules_wNP = uniqueMolecules_wNP.drop(columns=['NumSourceStarters', 'SourceStarters'])
uniqueMolecules_wNP

### How many pathways has been generated 

In [ ]:
import PyPDF2

# Count pathways from PDF files in each directory
pathwayCountsPerStarter = []

for starterDirPath in allStarterDirPaths_NP:
    dirName = os.path.basename(starterDirPath)
    starterNum = int(dirName.replace('starter_', '').replace('_NP', ''))

    expectedPdfPath = os.path.join(starterDirPath, f"{dirName}_pathways_visualized.pdf")

    if not os.path.exists(expectedPdfPath):
        fallbackPdfPaths = glob.glob(os.path.join(starterDirPath, "*_pathways_visualized.pdf"))
        expectedPdfPath = fallbackPdfPaths[0] if fallbackPdfPaths else None

    numPathways = 0
    if expectedPdfPath and os.path.exists(expectedPdfPath):
        try:
            with open(expectedPdfPath, 'rb') as pdfFile:
                pdfReader = PyPDF2.PdfReader(pdfFile)
                numPathways = len(pdfReader.pages)
        except Exception as e:
            print(f"Error reading PDF for {dirName}: {e}")

    pathwayCountsPerStarter.append({
        'StarterDirectory': dirName,
        'StarterNum': starterNum,
        'NumPathways': numPathways
    })

allPathways_wNPRaw = pd.DataFrame(pathwayCountsPerStarter).sort_values('StarterNum').reset_index(drop=True)

# Keep only starters with pathways
allPathways_wNP = allPathways_wNPRaw[allPathways_wNPRaw['NumPathways'] > 0].copy().reset_index(drop=True)

totalStarterDirs_NP = len(allStarterDirPaths_NP)
totalPathways = allPathways_wNP['NumPathways'].sum()
startersWithPathways = len(allPathways_wNP)
startersWithoutPathways = totalStarterDirs_NP - startersWithPathways
withPercent = (startersWithPathways / totalStarterDirs_NP) * 100
withoutPercent = (startersWithoutPathways / totalStarterDirs_NP) * 100

print(f"Total pathways found: {totalPathways}")
print(f"Starters with pathways: {startersWithPathways} ({withPercent:.2f}%), "
      f"without: {startersWithoutPathways} ({withoutPercent:.2f}%)")

allPathways_wNP = allPathways_wNP.drop(columns=['StarterNum'])
allPathways_wNP